In [ ]:
#earthquake
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# ==================== 1. Data loading and preprocessing ====================
def load_and_preprocess_data(file_path, outlier_method='remove', iqr_multiplier=3):
    """
    Load and preprocess the data.
    
    Parameters:
    - file_path: Path to the input data.
    - outlier_method: Use 'remove' to drop outliers or 'clip' to cap them.
    - iqr_multiplier: Multiplier for the outlier bounds (default: 3).
    """
    print("Loading data...")
    data = pd.read_csv(file_path)
    print(f"Original data shape: {data.shape}")
    print(f"Column names: {list(data.columns)}")
    
    # Check whether L(m) is present
    if 'L(m)' not in data.columns:
        raise ValueError("The data must contain an 'L(m)' column")
    
    # Handle missing values
    print(f"\nMissing values by column:\n{data.isnull().sum()}")
    initial_shape = data.shape
    data = data.dropna()
    print(f"Dropped {initial_shape[0] - data.shape[0]} rows with missing values")
    print(f"Data shape after dropping missing values: {data.shape}")
    
    # Keep a copy for comparison
    data_before_outlier = data.copy()
    
    # Detect outliers
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    print(f"\nOutlier detection with an IQR multiplier of {iqr_multiplier}:")
    print("-" * 50)
    
    if outlier_method == 'remove':
        # ========== Drop rows with outliers ==========
        # Create a Boolean mask for rows within the bounds
        mask = pd.Series([True] * len(data))
        outlier_details = {}
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # Identify outliers
            col_outliers_low = data[col] < lower_bound
            col_outliers_high = data[col] > upper_bound
            col_outliers = col_outliers_low | col_outliers_high
            
            # Count outliers
            n_outliers_low = col_outliers_low.sum()
            n_outliers_high = col_outliers_high.sum()
            n_outliers_total = col_outliers.sum()
            
            if n_outliers_total > 0:
                outlier_details[col] = {
                    'low': n_outliers_low,
                    'high': n_outliers_high,
                    'total': n_outliers_total,
                    'lower_bound': lower_bound,
                    'upper_bound': upper_bound
                }
                
                print(f"\nColumn '{col}':")
                print(f"  Range: [{data[col].min():.2f}, {data[col].max():.2f}]")
                print(f"  Outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
                print(f"  Outlier count: {n_outliers_total} ({n_outliers_total/len(data)*100:.1f}%)")
                print(f"    - Below the lower bound: {n_outliers_low}")
                print(f"    - Above the upper bound: {n_outliers_high}")
            
            # Keep rows within the bounds for every feature
            mask = mask & ~col_outliers
        
        # Apply the mask to remove outliers
        data_cleaned = data[mask].copy()
        n_removed = len(data) - len(data_cleaned)
        
        print(f"\n{'='*50}")
        print(f"Outlier removal summary:")
        print(f"  Original row count: {len(data)}")
        print(f"  Rows removed: {n_removed} ({n_removed/len(data)*100:.1f}%)")
        print(f"  Rows remaining: {len(data_cleaned)}")
        print(f"{'='*50}")
        
        data = data_cleaned
        
    else:
        # ========== Clip outliers ==========
        print("\nClipping outliers...")
        modified_count = 0
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # Count outliers
            outliers_low = (data[col] < lower_bound).sum()
            outliers_high = (data[col] > upper_bound).sum()
            
            if outliers_low > 0 or outliers_high > 0:
                print(f"\nColumn '{col}':")
                print(f"  - Values below the lower bound: {outliers_low}")
                print(f"  - Values above the upper bound: {outliers_high}")
                print(f"  - Bounds: [{lower_bound:.4f}, {upper_bound:.4f}]")
                modified_count += outliers_low + outliers_high
            
            # Clip outliers
            data[col] = data[col].clip(lower_bound, upper_bound)
        
        print(f"\nClipped {modified_count} outliers")
    
    print(f"\nFinal data shape: {data.shape}")
    
    # Compare statistics before and after preprocessing
    print("\n" + "="*50)
    print("Summary statistics comparison:")
    print("="*50)
    print("\nBefore preprocessing:")
    print(data_before_outlier.describe().round(2))
    print("\nAfter preprocessing:")
    print(data.describe().round(2))
    
    # Separate predictors from the target
    X = data.drop('L(m)', axis=1).values
    y = data['L(m)'].values.reshape(-1, 1)
    
    return X, y, data.columns.tolist()

# ==================== 2. Enhanced GAN architecture ====================
class FeatureAttention(nn.Module):
    """Feature attention mechanism."""
    def __init__(self, in_dim):
        super(FeatureAttention, self).__init__()
        self.fc1 = nn.Linear(in_dim, in_dim // 4)
        self.fc2 = nn.Linear(in_dim // 4, in_dim)
        self.sigmoid = nn.Sigmoid()
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        attn = self.fc1(x)
        attn = torch.relu(attn)
        attn = self.fc2(attn)
        attn = self.sigmoid(attn)
        attended = x * attn
        out = self.gamma * attended + x
        return out

class ResidualBlock(nn.Module):
    """Residual block."""
    def __init__(self, dim, dropout_rate=0.1):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim)
        )
        self.activation = nn.GELU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class EnhancedGenerator(nn.Module):
    """Enhanced generator network."""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], output_dim=1, dropout_rate=0.1):
        super(EnhancedGenerator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # Input projection
        layers.extend([
            nn.Linear(prev_dim, hidden_dims[0]),
            nn.LayerNorm(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        ])
        
        # Hidden layers
        for i in range(len(hidden_dims) - 1):
            layers.extend([
                nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                nn.LayerNorm(hidden_dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
            
            if i % 2 == 0:
                layers.append(ResidualBlock(hidden_dims[i+1], dropout_rate))
            
            if i == len(hidden_dims) // 2:
                layers.append(FeatureAttention(hidden_dims[i+1]))
        
        # Output layer
        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

class EnhancedDiscriminator(nn.Module):
    """Enhanced discriminator network."""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], dropout_rate=0.1):
        super(EnhancedDiscriminator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            if i < 2:
                layers.extend([
                    nn.utils.spectral_norm(nn.Linear(prev_dim, hidden_dim)),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            else:
                layers.extend([
                    nn.Linear(prev_dim, hidden_dim),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

# ==================== 3. Optimized GAN training ====================
class OptimizedGANRegressor:
    """Optimized GAN regression model."""
    def __init__(self, input_dim, device='cpu', weight_decay=5e-5):
        self.device = device
        self.input_dim = input_dim
        
        # Initialize the networks
        self.generator = EnhancedGenerator(input_dim).to(device)
        self.discriminator = EnhancedDiscriminator(input_dim + 1).to(device)
        
        # Optimizer
        self.g_optimizer = optim.AdamW(
            self.generator.parameters(), 
            lr=0.0002,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        self.d_optimizer = optim.AdamW(
            self.discriminator.parameters(), 
            lr=0.0001,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        
        # Learning-rate schedulers
        self.g_scheduler = CosineAnnealingWarmRestarts(self.g_optimizer, T_0=50, T_mult=2)
        self.d_scheduler = CosineAnnealingWarmRestarts(self.d_optimizer, T_0=50, T_mult=2)
        
        # Loss functions
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.regression_loss = nn.L1Loss()
        self.auxiliary_loss = nn.MSELoss()
        
        # Training history
        self.history = {
            'g_loss': [], 'd_loss': [], 'reg_loss': [],
            'train_r2': [], 'val_r2': []
        }
        
        # Early-stopping settings
        self.best_val_r2 = -float('inf')
        self.best_train_r2 = -float('inf')
        self.patience_counter = 0
        self.best_state = None
    
    def gradient_penalty(self, real_data, fake_data, lambda_gp=5):
        """Calculate the gradient penalty."""
        batch_size = real_data.size(0)
        alpha = torch.rand(batch_size, 1).to(self.device)
        alpha = alpha.expand_as(real_data)
        
        interpolated = alpha * real_data + (1 - alpha) * fake_data
        interpolated.requires_grad_(True)
        
        prob_interpolated = self.discriminator(interpolated)
        
        gradients = torch.autograd.grad(
            outputs=prob_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones_like(prob_interpolated),
            create_graph=True,
            retain_graph=True
        )[0]
        
        gradients = gradients.view(batch_size, -1)
        gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
        
        return gradient_penalty
    
    def train_step(self, real_x, real_y, epoch):
        """Run one training step."""
        batch_size = real_x.size(0)
        
        # Adjust label smoothing dynamically
        smooth_factor = max(0.8, 0.95 - epoch * 0.001)
        valid = torch.ones(batch_size, 1).to(self.device) * smooth_factor
        fake = torch.ones(batch_size, 1).to(self.device) * (1 - smooth_factor)
        
        # Train the discriminator
        self.d_optimizer.zero_grad()
        
        real_pairs = torch.cat([real_x, real_y], dim=1)
        real_pred = self.discriminator(real_pairs)
        d_real_loss = self.adversarial_loss(real_pred, valid)
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y.detach()], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        d_fake_loss = self.adversarial_loss(fake_pred, fake)
        
        gp = self.gradient_penalty(real_pairs, fake_pairs, lambda_gp=5)
        
        d_loss = (d_real_loss + d_fake_loss) / 2 + gp
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), 1.0)
        self.d_optimizer.step()
        
        # Train the generator
        self.g_optimizer.zero_grad()
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        
        g_adv_loss = self.adversarial_loss(fake_pred, valid)
        g_reg_loss_l1 = self.regression_loss(gen_y, real_y)
        g_reg_loss_mse = self.auxiliary_loss(gen_y, real_y)
        
        reg_weight = max(20, 30 - epoch * 0.05)
        
        g_loss = g_adv_loss + reg_weight * g_reg_loss_l1 + reg_weight * 0.5 * g_reg_loss_mse
        
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), 1.0)
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item(), g_reg_loss_l1.item()
    
    def fit(self, X_train, y_train, X_val, y_val, epochs=500, batch_size=32, patience=50, verbose=True):
        """Train the model."""
        self.generator.train()
        self.discriminator.train()
        
        # Create the data loader
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train), 
            torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        # Training loop
        for epoch in range(epochs):
            epoch_d_loss = []
            epoch_g_loss = []
            epoch_reg_loss = []
            
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                
                d_loss, g_loss, reg_loss = self.train_step(batch_x, batch_y, epoch)
                
                epoch_d_loss.append(d_loss)
                epoch_g_loss.append(g_loss)
                epoch_reg_loss.append(reg_loss)
            
            # Learning-rate schedule
            self.g_scheduler.step()
            self.d_scheduler.step()
            
            # Evaluate
            train_r2 = self.evaluate(X_train, y_train)
            val_r2 = self.evaluate(X_val, y_val)
            
            # Record training history
            self.history['d_loss'].append(np.mean(epoch_d_loss))
            self.history['g_loss'].append(np.mean(epoch_g_loss))
            self.history['reg_loss'].append(np.mean(epoch_reg_loss))
            self.history['train_r2'].append(train_r2)
            self.history['val_r2'].append(val_r2)
            
            # Save the best model
            combined_score = 0.4 * train_r2 + 0.6 * val_r2
            if combined_score > 0.4 * self.best_train_r2 + 0.6 * self.best_val_r2:
                self.best_val_r2 = val_r2
                self.best_train_r2 = train_r2
                self.patience_counter = 0
                self.best_state = {
                    'generator': self.generator.state_dict(),
                    'discriminator': self.discriminator.state_dict()
                }
            else:
                self.patience_counter += 1
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] "
                      f"D_loss: {np.mean(epoch_d_loss):.4f}, "
                      f"G_loss: {np.mean(epoch_g_loss):.4f}, "
                      f"Reg_loss: {np.mean(epoch_reg_loss):.4f}, "
                      f"Train_R2: {train_r2:.4f}, "
                      f"Val_R2: {val_r2:.4f}, "
                      f"Best_Train_R2: {self.best_train_r2:.4f}, "
                      f"Best_Val_R2: {self.best_val_r2:.4f}")
            
            # Early stopping
            if self.patience_counter >= patience and epoch > 100:
                print(f"\nEarly stopping triggered")
                print(f"Best training R²: {self.best_train_r2:.4f}")
                print(f"Best validation R²: {self.best_val_r2:.4f}")
                if self.best_state is not None:
                    self.generator.load_state_dict(self.best_state['generator'])
                    self.discriminator.load_state_dict(self.best_state['discriminator'])
                break
            
            # Continue training if training R² is still low
            if epoch == epochs - 1 and train_r2 < 0.6:
                print(f"\nTraining R² is still low ({train_r2:.4f}). Continuing training...")
                epochs = min(epochs + 100, 800)
    
    def predict(self, X):
        """Make predictions."""
        self.generator.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.generator(X_tensor).cpu().numpy()
        return predictions
    
    def evaluate(self, X, y):
        """Evaluate the model."""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 4. Ensemble wrapper ====================
class OptimizedEnsembleGAN:
    """Optimized GAN ensemble."""
    def __init__(self, n_models=3, input_dim=None, device='cpu'):
        self.n_models = n_models
        self.models = []
        self.input_dim = input_dim
        self.device = device
        self.weights = None
    
    def fit(self, X_train, y_train, X_val, y_val, **kwargs):
        """Train multiple models."""
        val_scores = []
        train_scores = []
        
        for i in range(self.n_models):
            print(f"\n{'='*50}")
            print(f"Training model {i+1}/{self.n_models}")
            print(f"{'='*50}")
            
            # Use a different seed for each model
            torch.manual_seed(42 + i * 10)
            np.random.seed(42 + i * 10)
            
            # Create and train the model
            model = OptimizedGANRegressor(
                input_dim=self.input_dim,
                device=self.device,
                weight_decay=5e-5 * (1 + i * 0.2)
            )
            
            # Train the model
            model.fit(X_train, y_train, X_val, y_val, **kwargs)
            
            # Evaluate
            train_score = model.evaluate(X_train, y_train)
            val_score = model.evaluate(X_val, y_val)
            
            train_scores.append(train_score)
            val_scores.append(val_score)
            self.models.append(model)
            
            print(f"\nResults for model {i+1}:")
            print(f"  Training R²: {train_score:.4f}")
            print(f"  Validation R²: {val_score:.4f}")
        
        # Calculate the averaging weights
        train_scores = np.array(train_scores)
        val_scores = np.array(val_scores)
        combined_scores = 0.4 * train_scores + 0.6 * val_scores
        
        self.weights = np.exp(combined_scores * 2) / np.sum(np.exp(combined_scores * 2))
        print(f"\nModel weights: {self.weights}")
    
    def predict(self, X):
        """Predict using the ensemble."""
        predictions = []
        for model in self.models:
            pred = model.predict(X)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        weighted_pred = np.average(predictions, axis=0, weights=self.weights)
        
        return weighted_pred
    
    def evaluate(self, X, y):
        """Evaluate the ensemble."""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 5. Model diagnostics and plots ====================
def diagnose_model(model, X_train, y_train, X_val, y_val, scaler_y):
    """Report model diagnostics."""
    print("\n" + "="*50)
    print("Model diagnostics")
    print("="*50)
    
    # Training performance
    y_train_pred = model.predict(X_train)
    y_train_pred_original = scaler_y.inverse_transform(y_train_pred)
    y_train_original = scaler_y.inverse_transform(y_train)
    
    train_r2 = r2_score(y_train_original, y_train_pred_original)
    train_mse = mean_squared_error(y_train_original, y_train_pred_original)
    train_mae = mean_absolute_error(y_train_original, y_train_pred_original)
    
    print(f"\nTraining performance (original scale):")
    print(f"  R² Score: {train_r2:.4f}")
    print(f"  MSE: {train_mse:.4f}")
    print(f"  MAE: {train_mae:.4f}")
    print(f"  RMSE: {np.sqrt(train_mse):.4f}")
    
    # Validation performance
    y_val_pred = model.predict(X_val)
    y_val_pred_original = scaler_y.inverse_transform(y_val_pred)
    y_val_original = scaler_y.inverse_transform(y_val)
    
    val_r2 = r2_score(y_val_original, y_val_pred_original)
    val_mse = mean_squared_error(y_val_original, y_val_pred_original)
    val_mae = mean_absolute_error(y_val_original, y_val_pred_original)
    
    print(f"\nValidation performance (original scale):")
    print(f"  R² Score: {val_r2:.4f}")
    print(f"  MSE: {val_mse:.4f}")
    print(f"  MAE: {val_mae:.4f}")
    print(f"  RMSE: {np.sqrt(val_mse):.4f}")
    
    # Summarize the diagnostics
    print("\n" + "-"*50)
    print("Diagnostic summary:")
    
    r2_diff = abs(train_r2 - val_r2)
    
    if train_r2 < 0.6:
        if val_r2 > 0.8:
            print("⚠️ Low training performance and good validation performance")
            print("  The training set may contain more noise or outliers")
        else:
            print("⚠️ Possible underfitting")
            print(f"  - Low training R²: {train_r2:.4f}")
    elif r2_diff > 0.15:
        print("⚠️ Possible overfitting")
        print(f"  - Training–validation R² gap: {r2_diff:.4f}")
    else:
        print("✅ Good model performance")
        print(f"  - Training R²: {train_r2:.4f}")
        print(f"  - Validation R²: {val_r2:.4f}")
        print(f"  - Difference: {r2_diff:.4f}")
    
    # Check whether the model meets the targets
    if val_r2 >= 0.8 and train_r2 >= 0.6:
        print("\n🎉 Targets met!")
        print(f"  - Validation R² >= 0.8: {val_r2:.4f}")
        print(f"  - Training R² >= 0.6: {train_r2:.4f}")
    elif val_r2 >= 0.8:
        print(f"\n⚠️ Validation met its target (R²={val_r2:.4f}). Training needs improvement (R²={train_r2:.4f}).")
    else:
        print(f"\n❌ Validation target missed (R²={val_r2:.4f} < 0.8)")
    
    return {
        'train_r2': train_r2, 'val_r2': val_r2,
        'train_mse': train_mse, 'val_mse': val_mse,
        'y_train_pred': y_train_pred_original,
        'y_val_pred': y_val_pred_original
    }

def plot_results(history, results, y_train_original, y_val_original):
    """Plot the results."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Read the training history
    if isinstance(history, list):
        plot_history = history[0]
    elif hasattr(history, 'models') and history.models:
        plot_history = history.models[0].history
    else:
        plot_history = history.history if hasattr(history, 'history') else history
    
    # 1. Loss curves
    ax = axes[0, 0]
    ax.plot(plot_history['d_loss'], label='Discriminator Loss', alpha=0.7)
    ax.plot(plot_history['g_loss'], label='Generator Loss', alpha=0.7)
    ax.plot(plot_history['reg_loss'], label='Regression Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training losses')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. R² curves
    ax = axes[0, 1]
    ax.plot(plot_history['train_r2'], label='Train R²', marker='o', markersize=2)
    ax.plot(plot_history['val_r2'], label='Validation R²', marker='s', markersize=2)
    ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target R²=0.8')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Min Train R²=0.6')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('R² Score')
    ax.set_title('R² over epochs')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. Training predictions vs. observations
    ax = axes[0, 2]
    ax.scatter(y_train_original, results['y_train_pred'], alpha=0.5, s=10)
    ax.plot([y_train_original.min(), y_train_original.max()], 
            [y_train_original.min(), y_train_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('Observed runout distance (m)')
    ax.set_ylabel('Predicted runout distance (m)')
    ax.set_title(f'Training set: R² = {results["train_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 4. Validation predictions vs. observations
    ax = axes[1, 0]
    ax.scatter(y_val_original, results['y_val_pred'], alpha=0.5, s=10, color='orange')
    ax.plot([y_val_original.min(), y_val_original.max()], 
            [y_val_original.min(), y_val_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('Observed runout distance (m)')
    ax.set_ylabel('Predicted runout distance (m)')
    ax.set_title(f'Validation set: R² = {results["val_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 5. Training residual distribution
    ax = axes[1, 1]
    residuals_train = y_train_original.flatten() - results['y_train_pred'].flatten()
    ax.hist(residuals_train, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('Residual (m)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Training residuals (mean={np.mean(residuals_train):.2f})')
    ax.grid(True, alpha=0.3)
    
    # 6. Validation residual distribution
    ax = axes[1, 2]
    residuals_val = y_val_original.flatten() - results['y_val_pred'].flatten()
    ax.hist(residuals_val, bins=30, edgecolor='black', alpha=0.7, color='orange')
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('Residual (m)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Validation residuals (mean={np.mean(residuals_val):.2f})')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('gan_outlier_removal_results.png', dpi=100, bbox_inches='tight')
    plt.show()

# ==================== 6. Main function ====================
def main(outlier_method='remove', iqr_multiplier=3, use_ensemble=True):
    """
    Run the runout distance prediction workflow.
    
    Parameters:
    - outlier_method: Choose 'remove' to drop outliers or 'clip' to cap them.
    - iqr_multiplier: Multiplier for the outlier bounds.
    - use_ensemble: Whether to train an ensemble.
    """
    # Select the computation device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    print(f"Outlier treatment: {outlier_method}")
    print(f"IQR multiplier: {iqr_multiplier}")
    print(f"Use ensemble: {use_ensemble}")
    print("="*50)
    
    # 1. Load the data and treat outliers
    X, y, column_names = load_and_preprocess_data(
        "imputed_earthquake_with_runout.csv",
        outlier_method=outlier_method,
        iqr_multiplier=iqr_multiplier
    )
    
    # 2. Scale the data
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)
    
    # 3. Split the training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.2, random_state=42
    )
    
    print(f"\nData split:")
    print(f"Training set shape: {X_train.shape}")
    print(f"Validation set shape: {X_val.shape}")
    print(f"Feature count: {X_train.shape[1]}")
    
    # 4. Select the model type
    if use_ensemble:
        print("\nTraining the optimized GAN ensemble...")
        model = OptimizedEnsembleGAN(
            n_models=3, 
            input_dim=X_train.shape[1],
            device=device
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    else:
        print("\nTraining a single optimized GAN...")
        model = OptimizedGANRegressor(
            input_dim=X_train.shape[1],
            device=device,
            weight_decay=5e-5
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    
    # 5. Diagnose model performance
    results = diagnose_model(
        model, 
        X_train, y_train, 
        X_val, y_val,
        scaler_y
    )
    
    # 6. Plot the results
    y_train_original = scaler_y.inverse_transform(y_train)
    y_val_original = scaler_y.inverse_transform(y_val)
    plot_results(model, results, y_train_original, y_val_original)
    
    # 7. Report the final results
    print("\n" + "="*50)
    print("Final results")
    print("="*50)
    print(f"Outlier treatment: {outlier_method}")
    print(f"Rows after preprocessing: {X.shape[0]}")
    print(f"\nR² scores on the original scale:")
    print(f"  Training: {results['train_r2']:.4f}")
    print(f"  Validation: {results['val_r2']:.4f}")
    print(f"  Difference: {abs(results['train_r2'] - results['val_r2']):.4f}")
    # 8. Export predictions to CSV
    train_df = pd.DataFrame({
        "Dataset": "Train",
        "Actual": y_train_original.flatten(),
        "Predicted": results['y_train_pred'].flatten()
    })

    val_df = pd.DataFrame({
        "Dataset": "Validation",
        "Actual": y_val_original.flatten(),
        "Predicted": results['y_val_pred'].flatten()
    })

    output_df = pd.concat([train_df, val_df], ignore_index=True)
    output_df.to_csv("train_val_predictions.csv", index=False, encoding="utf-8-sig")
    print("\n✅ Predictions saved to train_val_predictions.csv")
    
    if results['val_r2'] >= 0.8 and results['train_r2'] >= 0.6:
        print("\n🎉 Both performance targets met!")
        print(f"  Validation R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  Training R² = {results['train_r2']:.4f} >= 0.6 ✓")
        print("  The model meets both configured R² targets")
    elif results['val_r2'] >= 0.8:
        print(f"\n⚠️ Only the validation target was met")
        print(f"  Validation R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  Training R² = {results['train_r2']:.4f} < 0.6 ✗")
        print("\nSuggestions:")
        print("  1. Check the quality of the training data")
        print("  2. Consider data cleaning or feature engineering")
        print("  3. Adjust the model architecture or training strategy")
    else:
        print(f"\n❌ Validation target missed")
        print(f"  Validation R² = {results['val_r2']:.4f} < 0.8")
        print("\nSuggestions:")
        print("  1. Increase model capacity")
        print("  2. Collect more training data")
        print("  3. Try a different model architecture")
    
    return model, results

# Run the workflow
if __name__ == "__main__":
    # Adjustable parameters:
    # outlier_method: 'remove' to drop outliers or 'clip' to cap them
    # iqr_multiplier: 1.5 (strict), 3 (standard), or 5 (lenient)
    # use_ensemble: True for an ensemble or False for a single model
    
    model, results = main(
        outlier_method='clip',  # Clip outliers
        iqr_multiplier=1.5,         # Use an IQR multiplier of 1.5
        use_ensemble=True         # Use the ensemble
    )
    
    print("\nTraining complete. Figure saved as 'gan_outlier_removal_results.png'.")

In [ ]:
#mixed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# ==================== 1. Data loading and preprocessing ====================
def load_and_preprocess_data(file_path, outlier_method='remove', iqr_multiplier=3):
    """
    Load and preprocess the data.
    
    Parameters:
    - file_path: Path to the input data.
    - outlier_method: Use 'remove' to drop outliers or 'clip' to cap them.
    - iqr_multiplier: Multiplier for the outlier bounds (default: 3).
    """
    print("Loading data...")
    data = pd.read_csv(file_path)
    print(f"Original data shape: {data.shape}")
    print(f"Column names: {list(data.columns)}")
    
    # Check whether L(m) is present
    if 'L(m)' not in data.columns:
        raise ValueError("The data must contain an 'L(m)' column")
    
    # Handle missing values
    print(f"\nMissing values by column:\n{data.isnull().sum()}")
    initial_shape = data.shape
    data = data.dropna()
    print(f"Dropped {initial_shape[0] - data.shape[0]} rows with missing values")
    print(f"Data shape after dropping missing values: {data.shape}")
    
    # Keep a copy for comparison
    data_before_outlier = data.copy()
    
    # Detect outliers
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    print(f"\nOutlier detection with an IQR multiplier of {iqr_multiplier}:")
    print("-" * 50)
    
    if outlier_method == 'remove':
        # ========== Drop rows with outliers ==========
        # Create a Boolean mask for rows within the bounds
        mask = pd.Series([True] * len(data))
        outlier_details = {}
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # Identify outliers
            col_outliers_low = data[col] < lower_bound
            col_outliers_high = data[col] > upper_bound
            col_outliers = col_outliers_low | col_outliers_high
            
            # Count outliers
            n_outliers_low = col_outliers_low.sum()
            n_outliers_high = col_outliers_high.sum()
            n_outliers_total = col_outliers.sum()
            
            if n_outliers_total > 0:
                outlier_details[col] = {
                    'low': n_outliers_low,
                    'high': n_outliers_high,
                    'total': n_outliers_total,
                    'lower_bound': lower_bound,
                    'upper_bound': upper_bound
                }
                
                print(f"\nColumn '{col}':")
                print(f"  Range: [{data[col].min():.2f}, {data[col].max():.2f}]")
                print(f"  Outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
                print(f"  Outlier count: {n_outliers_total} ({n_outliers_total/len(data)*100:.1f}%)")
                print(f"    - Below the lower bound: {n_outliers_low}")
                print(f"    - Above the upper bound: {n_outliers_high}")
            
            # Keep rows within the bounds for every feature
            mask = mask & ~col_outliers
        
        # Apply the mask to remove outliers
        data_cleaned = data[mask].copy()
        n_removed = len(data) - len(data_cleaned)
        
        print(f"\n{'='*50}")
        print(f"Outlier removal summary:")
        print(f"  Original row count: {len(data)}")
        print(f"  Rows removed: {n_removed} ({n_removed/len(data)*100:.1f}%)")
        print(f"  Rows remaining: {len(data_cleaned)}")
        print(f"{'='*50}")
        
        data = data_cleaned
        
    else:
        # ========== Clip outliers ==========
        print("\nClipping outliers...")
        modified_count = 0
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # Count outliers
            outliers_low = (data[col] < lower_bound).sum()
            outliers_high = (data[col] > upper_bound).sum()
            
            if outliers_low > 0 or outliers_high > 0:
                print(f"\nColumn '{col}':")
                print(f"  - Values below the lower bound: {outliers_low}")
                print(f"  - Values above the upper bound: {outliers_high}")
                print(f"  - Bounds: [{lower_bound:.4f}, {upper_bound:.4f}]")
                modified_count += outliers_low + outliers_high
            
            # Clip outliers
            data[col] = data[col].clip(lower_bound, upper_bound)
        
        print(f"\nClipped {modified_count} outliers")
    
    print(f"\nFinal data shape: {data.shape}")
    
    # Compare statistics before and after preprocessing
    print("\n" + "="*50)
    print("Summary statistics comparison:")
    print("="*50)
    print("\nBefore preprocessing:")
    print(data_before_outlier.describe().round(2))
    print("\nAfter preprocessing:")
    print(data.describe().round(2))
    
    # Separate predictors from the target
    X = data.drop('L(m)', axis=1).values
    y = data['L(m)'].values.reshape(-1, 1)
    
    return X, y, data.columns.tolist()

# ==================== 2. Enhanced GAN architecture ====================
class FeatureAttention(nn.Module):
    """Feature attention mechanism."""
    def __init__(self, in_dim):
        super(FeatureAttention, self).__init__()
        self.fc1 = nn.Linear(in_dim, in_dim // 4)
        self.fc2 = nn.Linear(in_dim // 4, in_dim)
        self.sigmoid = nn.Sigmoid()
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        attn = self.fc1(x)
        attn = torch.relu(attn)
        attn = self.fc2(attn)
        attn = self.sigmoid(attn)
        attended = x * attn
        out = self.gamma * attended + x
        return out

class ResidualBlock(nn.Module):
    """Residual block."""
    def __init__(self, dim, dropout_rate=0.1):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim)
        )
        self.activation = nn.GELU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class EnhancedGenerator(nn.Module):
    """Enhanced generator network."""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], output_dim=1, dropout_rate=0.1):
        super(EnhancedGenerator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # Input projection
        layers.extend([
            nn.Linear(prev_dim, hidden_dims[0]),
            nn.LayerNorm(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        ])
        
        # Hidden layers
        for i in range(len(hidden_dims) - 1):
            layers.extend([
                nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                nn.LayerNorm(hidden_dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
            
            if i % 2 == 0:
                layers.append(ResidualBlock(hidden_dims[i+1], dropout_rate))
            
            if i == len(hidden_dims) // 2:
                layers.append(FeatureAttention(hidden_dims[i+1]))
        
        # Output layer
        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

class EnhancedDiscriminator(nn.Module):
    """Enhanced discriminator network."""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], dropout_rate=0.1):
        super(EnhancedDiscriminator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            if i < 2:
                layers.extend([
                    nn.utils.spectral_norm(nn.Linear(prev_dim, hidden_dim)),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            else:
                layers.extend([
                    nn.Linear(prev_dim, hidden_dim),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

# ==================== 3. Optimized GAN training ====================
class OptimizedGANRegressor:
    """Optimized GAN regression model."""
    def __init__(self, input_dim, device='cpu', weight_decay=5e-5):
        self.device = device
        self.input_dim = input_dim
        
        # Initialize the networks
        self.generator = EnhancedGenerator(input_dim).to(device)
        self.discriminator = EnhancedDiscriminator(input_dim + 1).to(device)
        
        # Optimizer
        self.g_optimizer = optim.AdamW(
            self.generator.parameters(), 
            lr=0.0002,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        self.d_optimizer = optim.AdamW(
            self.discriminator.parameters(), 
            lr=0.0001,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        
        # Learning-rate schedulers
        self.g_scheduler = CosineAnnealingWarmRestarts(self.g_optimizer, T_0=50, T_mult=2)
        self.d_scheduler = CosineAnnealingWarmRestarts(self.d_optimizer, T_0=50, T_mult=2)
        
        # Loss functions
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.regression_loss = nn.L1Loss()
        self.auxiliary_loss = nn.MSELoss()
        
        # Training history
        self.history = {
            'g_loss': [], 'd_loss': [], 'reg_loss': [],
            'train_r2': [], 'val_r2': []
        }
        
        # Early-stopping settings
        self.best_val_r2 = -float('inf')
        self.best_train_r2 = -float('inf')
        self.patience_counter = 0
        self.best_state = None
    
    def gradient_penalty(self, real_data, fake_data, lambda_gp=5):
        """Calculate the gradient penalty."""
        batch_size = real_data.size(0)
        alpha = torch.rand(batch_size, 1).to(self.device)
        alpha = alpha.expand_as(real_data)
        
        interpolated = alpha * real_data + (1 - alpha) * fake_data
        interpolated.requires_grad_(True)
        
        prob_interpolated = self.discriminator(interpolated)
        
        gradients = torch.autograd.grad(
            outputs=prob_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones_like(prob_interpolated),
            create_graph=True,
            retain_graph=True
        )[0]
        
        gradients = gradients.view(batch_size, -1)
        gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
        
        return gradient_penalty
    
    def train_step(self, real_x, real_y, epoch):
        """Run one training step."""
        batch_size = real_x.size(0)
        
        # Adjust label smoothing dynamically
        smooth_factor = max(0.8, 0.95 - epoch * 0.001)
        valid = torch.ones(batch_size, 1).to(self.device) * smooth_factor
        fake = torch.ones(batch_size, 1).to(self.device) * (1 - smooth_factor)
        
        # Train the discriminator
        self.d_optimizer.zero_grad()
        
        real_pairs = torch.cat([real_x, real_y], dim=1)
        real_pred = self.discriminator(real_pairs)
        d_real_loss = self.adversarial_loss(real_pred, valid)
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y.detach()], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        d_fake_loss = self.adversarial_loss(fake_pred, fake)
        
        gp = self.gradient_penalty(real_pairs, fake_pairs, lambda_gp=5)
        
        d_loss = (d_real_loss + d_fake_loss) / 2 + gp
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), 1.0)
        self.d_optimizer.step()
        
        # Train the generator
        self.g_optimizer.zero_grad()
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        
        g_adv_loss = self.adversarial_loss(fake_pred, valid)
        g_reg_loss_l1 = self.regression_loss(gen_y, real_y)
        g_reg_loss_mse = self.auxiliary_loss(gen_y, real_y)
        
        reg_weight = max(20, 30 - epoch * 0.05)
        
        g_loss = g_adv_loss + reg_weight * g_reg_loss_l1 + reg_weight * 0.5 * g_reg_loss_mse
        
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), 1.0)
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item(), g_reg_loss_l1.item()
    
    def fit(self, X_train, y_train, X_val, y_val, epochs=500, batch_size=32, patience=50, verbose=True):
        """Train the model."""
        self.generator.train()
        self.discriminator.train()
        
        # Create the data loader
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train), 
            torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        # Training loop
        for epoch in range(epochs):
            epoch_d_loss = []
            epoch_g_loss = []
            epoch_reg_loss = []
            
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                
                d_loss, g_loss, reg_loss = self.train_step(batch_x, batch_y, epoch)
                
                epoch_d_loss.append(d_loss)
                epoch_g_loss.append(g_loss)
                epoch_reg_loss.append(reg_loss)
            
            # Learning-rate schedule
            self.g_scheduler.step()
            self.d_scheduler.step()
            
            # Evaluate
            train_r2 = self.evaluate(X_train, y_train)
            val_r2 = self.evaluate(X_val, y_val)
            
            # Record training history
            self.history['d_loss'].append(np.mean(epoch_d_loss))
            self.history['g_loss'].append(np.mean(epoch_g_loss))
            self.history['reg_loss'].append(np.mean(epoch_reg_loss))
            self.history['train_r2'].append(train_r2)
            self.history['val_r2'].append(val_r2)
            
            # Save the best model
            combined_score = 0.4 * train_r2 + 0.6 * val_r2
            if combined_score > 0.4 * self.best_train_r2 + 0.6 * self.best_val_r2:
                self.best_val_r2 = val_r2
                self.best_train_r2 = train_r2
                self.patience_counter = 0
                self.best_state = {
                    'generator': self.generator.state_dict(),
                    'discriminator': self.discriminator.state_dict()
                }
            else:
                self.patience_counter += 1
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] "
                      f"D_loss: {np.mean(epoch_d_loss):.4f}, "
                      f"G_loss: {np.mean(epoch_g_loss):.4f}, "
                      f"Reg_loss: {np.mean(epoch_reg_loss):.4f}, "
                      f"Train_R2: {train_r2:.4f}, "
                      f"Val_R2: {val_r2:.4f}, "
                      f"Best_Train_R2: {self.best_train_r2:.4f}, "
                      f"Best_Val_R2: {self.best_val_r2:.4f}")
            
            # Early stopping
            if self.patience_counter >= patience and epoch > 100:
                print(f"\nEarly stopping triggered")
                print(f"Best training R²: {self.best_train_r2:.4f}")
                print(f"Best validation R²: {self.best_val_r2:.4f}")
                if self.best_state is not None:
                    self.generator.load_state_dict(self.best_state['generator'])
                    self.discriminator.load_state_dict(self.best_state['discriminator'])
                break
            
            # Continue training if training R² is still low
            if epoch == epochs - 1 and train_r2 < 0.6:
                print(f"\nTraining R² is still low ({train_r2:.4f}). Continuing training...")
                epochs = min(epochs + 100, 800)
    
    def predict(self, X):
        """Make predictions."""
        self.generator.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.generator(X_tensor).cpu().numpy()
        return predictions
    
    def evaluate(self, X, y):
        """Evaluate the model."""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 4. Ensemble wrapper ====================
class OptimizedEnsembleGAN:
    """Optimized GAN ensemble."""
    def __init__(self, n_models=3, input_dim=None, device='cpu'):
        self.n_models = n_models
        self.models = []
        self.input_dim = input_dim
        self.device = device
        self.weights = None
    
    def fit(self, X_train, y_train, X_val, y_val, **kwargs):
        """Train multiple models."""
        val_scores = []
        train_scores = []
        
        for i in range(self.n_models):
            print(f"\n{'='*50}")
            print(f"Training model {i+1}/{self.n_models}")
            print(f"{'='*50}")
            
            # Use a different seed for each model
            torch.manual_seed(42 + i * 10)
            np.random.seed(42 + i * 10)
            
            # Create and train the model
            model = OptimizedGANRegressor(
                input_dim=self.input_dim,
                device=self.device,
                weight_decay=5e-5 * (1 + i * 0.2)
            )
            
            # Train the model
            model.fit(X_train, y_train, X_val, y_val, **kwargs)
            
            # Evaluate
            train_score = model.evaluate(X_train, y_train)
            val_score = model.evaluate(X_val, y_val)
            
            train_scores.append(train_score)
            val_scores.append(val_score)
            self.models.append(model)
            
            print(f"\nResults for model {i+1}:")
            print(f"  Training R²: {train_score:.4f}")
            print(f"  Validation R²: {val_score:.4f}")
        
        # Calculate the averaging weights
        train_scores = np.array(train_scores)
        val_scores = np.array(val_scores)
        combined_scores = 0.4 * train_scores + 0.6 * val_scores
        
        self.weights = np.exp(combined_scores * 2) / np.sum(np.exp(combined_scores * 2))
        print(f"\nModel weights: {self.weights}")
    
    def predict(self, X):
        """Predict using the ensemble."""
        predictions = []
        for model in self.models:
            pred = model.predict(X)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        weighted_pred = np.average(predictions, axis=0, weights=self.weights)
        
        return weighted_pred
    
    def evaluate(self, X, y):
        """Evaluate the ensemble."""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 5. Model diagnostics and plots ====================
def diagnose_model(model, X_train, y_train, X_val, y_val, scaler_y):
    """Report model diagnostics."""
    print("\n" + "="*50)
    print("Model diagnostics")
    print("="*50)
    
    # Training performance
    y_train_pred = model.predict(X_train)
    y_train_pred_original = scaler_y.inverse_transform(y_train_pred)
    y_train_original = scaler_y.inverse_transform(y_train)
    
    train_r2 = r2_score(y_train_original, y_train_pred_original)
    train_mse = mean_squared_error(y_train_original, y_train_pred_original)
    train_mae = mean_absolute_error(y_train_original, y_train_pred_original)
    
    print(f"\nTraining performance (original scale):")
    print(f"  R² Score: {train_r2:.4f}")
    print(f"  MSE: {train_mse:.4f}")
    print(f"  MAE: {train_mae:.4f}")
    print(f"  RMSE: {np.sqrt(train_mse):.4f}")
    
    # Validation performance
    y_val_pred = model.predict(X_val)
    y_val_pred_original = scaler_y.inverse_transform(y_val_pred)
    y_val_original = scaler_y.inverse_transform(y_val)
    
    val_r2 = r2_score(y_val_original, y_val_pred_original)
    val_mse = mean_squared_error(y_val_original, y_val_pred_original)
    val_mae = mean_absolute_error(y_val_original, y_val_pred_original)
    
    print(f"\nValidation performance (original scale):")
    print(f"  R² Score: {val_r2:.4f}")
    print(f"  MSE: {val_mse:.4f}")
    print(f"  MAE: {val_mae:.4f}")
    print(f"  RMSE: {np.sqrt(val_mse):.4f}")
    
    # Summarize the diagnostics
    print("\n" + "-"*50)
    print("Diagnostic summary:")
    
    r2_diff = abs(train_r2 - val_r2)
    
    if train_r2 < 0.6:
        if val_r2 > 0.8:
            print("⚠️ Low training performance and good validation performance")
            print("  The training set may contain more noise or outliers")
        else:
            print("⚠️ Possible underfitting")
            print(f"  - Low training R²: {train_r2:.4f}")
    elif r2_diff > 0.15:
        print("⚠️ Possible overfitting")
        print(f"  - Training–validation R² gap: {r2_diff:.4f}")
    else:
        print("✅ Good model performance")
        print(f"  - Training R²: {train_r2:.4f}")
        print(f"  - Validation R²: {val_r2:.4f}")
        print(f"  - Difference: {r2_diff:.4f}")
    
    # Check whether the model meets the targets
    if val_r2 >= 0.8 and train_r2 >= 0.6:
        print("\n🎉 Targets met!")
        print(f"  - Validation R² >= 0.8: {val_r2:.4f}")
        print(f"  - Training R² >= 0.6: {train_r2:.4f}")
    elif val_r2 >= 0.8:
        print(f"\n⚠️ Validation met its target (R²={val_r2:.4f}). Training needs improvement (R²={train_r2:.4f}).")
    else:
        print(f"\n❌ Validation target missed (R²={val_r2:.4f} < 0.8)")
    
    return {
        'train_r2': train_r2, 'val_r2': val_r2,
        'train_mse': train_mse, 'val_mse': val_mse,
        'y_train_pred': y_train_pred_original,
        'y_val_pred': y_val_pred_original
    }

def plot_results(history, results, y_train_original, y_val_original):
    """Plot the results."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Read the training history
    if isinstance(history, list):
        plot_history = history[0]
    elif hasattr(history, 'models') and history.models:
        plot_history = history.models[0].history
    else:
        plot_history = history.history if hasattr(history, 'history') else history
    
    # 1. Loss curves
    ax = axes[0, 0]
    ax.plot(plot_history['d_loss'], label='Discriminator Loss', alpha=0.7)
    ax.plot(plot_history['g_loss'], label='Generator Loss', alpha=0.7)
    ax.plot(plot_history['reg_loss'], label='Regression Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training losses')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. R² curves
    ax = axes[0, 1]
    ax.plot(plot_history['train_r2'], label='Train R²', marker='o', markersize=2)
    ax.plot(plot_history['val_r2'], label='Validation R²', marker='s', markersize=2)
    ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target R²=0.8')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Min Train R²=0.6')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('R² Score')
    ax.set_title('R² over epochs')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. Training predictions vs. observations
    ax = axes[0, 2]
    ax.scatter(y_train_original, results['y_train_pred'], alpha=0.5, s=10)
    ax.plot([y_train_original.min(), y_train_original.max()], 
            [y_train_original.min(), y_train_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('Observed runout distance (m)')
    ax.set_ylabel('Predicted runout distance (m)')
    ax.set_title(f'Training set: R² = {results["train_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 4. Validation predictions vs. observations
    ax = axes[1, 0]
    ax.scatter(y_val_original, results['y_val_pred'], alpha=0.5, s=10, color='orange')
    ax.plot([y_val_original.min(), y_val_original.max()], 
            [y_val_original.min(), y_val_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('Observed runout distance (m)')
    ax.set_ylabel('Predicted runout distance (m)')
    ax.set_title(f'Validation set: R² = {results["val_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 5. Training residual distribution
    ax = axes[1, 1]
    residuals_train = y_train_original.flatten() - results['y_train_pred'].flatten()
    ax.hist(residuals_train, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('Residual (m)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Training residuals (mean={np.mean(residuals_train):.2f})')
    ax.grid(True, alpha=0.3)
    
    # 6. Validation residual distribution
    ax = axes[1, 2]
    residuals_val = y_val_original.flatten() - results['y_val_pred'].flatten()
    ax.hist(residuals_val, bins=30, edgecolor='black', alpha=0.7, color='orange')
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('Residual (m)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Validation residuals (mean={np.mean(residuals_val):.2f})')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('gan_outlier_removal_results.png', dpi=100, bbox_inches='tight')
    plt.show()

# ==================== 6. Main function ====================
def main(outlier_method='remove', iqr_multiplier=3, use_ensemble=True):
    """
    Run the runout distance prediction workflow.
    
    Parameters:
    - outlier_method: Choose 'remove' to drop outliers or 'clip' to cap them.
    - iqr_multiplier: Multiplier for the outlier bounds.
    - use_ensemble: Whether to train an ensemble.
    """
    # Select the computation device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    print(f"Outlier treatment: {outlier_method}")
    print(f"IQR multiplier: {iqr_multiplier}")
    print(f"Use ensemble: {use_ensemble}")
    print("="*50)
    
    # 1. Load the data and treat outliers
    X, y, column_names = load_and_preprocess_data(
        "imputed_mixed_with_runout.csv",
        outlier_method=outlier_method,
        iqr_multiplier=iqr_multiplier
    )
    
    # 2. Scale the data
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)
    
    # 3. Split the training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.2, random_state=42
    )
    
    print(f"\nData split:")
    print(f"Training set shape: {X_train.shape}")
    print(f"Validation set shape: {X_val.shape}")
    print(f"Feature count: {X_train.shape[1]}")
    
    # 4. Select the model type
    if use_ensemble:
        print("\nTraining the optimized GAN ensemble...")
        model = OptimizedEnsembleGAN(
            n_models=3, 
            input_dim=X_train.shape[1],
            device=device
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    else:
        print("\nTraining a single optimized GAN...")
        model = OptimizedGANRegressor(
            input_dim=X_train.shape[1],
            device=device,
            weight_decay=5e-5
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    
    # 5. Diagnose model performance
    results = diagnose_model(
        model, 
        X_train, y_train, 
        X_val, y_val,
        scaler_y
    )
    
    # 6. Plot the results
    y_train_original = scaler_y.inverse_transform(y_train)
    y_val_original = scaler_y.inverse_transform(y_val)
    plot_results(model, results, y_train_original, y_val_original)
    
    # 7. Report the final results
    print("\n" + "="*50)
    print("Final results")
    print("="*50)
    print(f"Outlier treatment: {outlier_method}")
    print(f"Rows after preprocessing: {X.shape[0]}")
    print(f"\nR² scores on the original scale:")
    print(f"  Training: {results['train_r2']:.4f}")
    print(f"  Validation: {results['val_r2']:.4f}")
    print(f"  Difference: {abs(results['train_r2'] - results['val_r2']):.4f}")
    # 8. Export predictions to CSV
    train_df = pd.DataFrame({
        "Dataset": "Train",
        "Actual": y_train_original.flatten(),
        "Predicted": results['y_train_pred'].flatten()
    })

    val_df = pd.DataFrame({
        "Dataset": "Validation",
        "Actual": y_val_original.flatten(),
        "Predicted": results['y_val_pred'].flatten()
    })

    output_df = pd.concat([train_df, val_df], ignore_index=True)
    output_df.to_csv("train_val_predictions.csv", index=False, encoding="utf-8-sig")
    print("\n✅ Predictions saved to train_val_predictions.csv")
    
    if results['val_r2'] >= 0.8 and results['train_r2'] >= 0.6:
        print("\n🎉 Both performance targets met!")
        print(f"  Validation R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  Training R² = {results['train_r2']:.4f} >= 0.6 ✓")
        print("  The model meets both configured R² targets")
    elif results['val_r2'] >= 0.8:
        print(f"\n⚠️ Only the validation target was met")
        print(f"  Validation R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  Training R² = {results['train_r2']:.4f} < 0.6 ✗")
        print("\nSuggestions:")
        print("  1. Check the quality of the training data")
        print("  2. Consider data cleaning or feature engineering")
        print("  3. Adjust the model architecture or training strategy")
    else:
        print(f"\n❌ Validation target missed")
        print(f"  Validation R² = {results['val_r2']:.4f} < 0.8")
        print("\nSuggestions:")
        print("  1. Increase model capacity")
        print("  2. Collect more training data")
        print("  3. Try a different model architecture")
    
    return model, results

# Run the workflow
if __name__ == "__main__":
    # Adjustable parameters:
    # outlier_method: 'remove' to drop outliers or 'clip' to cap them
    # iqr_multiplier: 1.5 (strict), 3 (standard), or 5 (lenient)
    # use_ensemble: True for an ensemble or False for a single model
    
    model, results = main(
        outlier_method='clip',  # Clip outliers
        iqr_multiplier=1.5,         # Use an IQR multiplier of 1.5
        use_ensemble=True         # Use the ensemble
    )
    
    print("\nTraining complete. Figure saved as 'gan_outlier_removal_results.png'.")

In [ ]:
#others
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# ==================== 1. Data loading and preprocessing ====================
def load_and_preprocess_data(file_path, outlier_method='remove', iqr_multiplier=3):
    """
    Load and preprocess the data.
    
    Parameters:
    - file_path: Path to the input data.
    - outlier_method: Use 'remove' to drop outliers or 'clip' to cap them.
    - iqr_multiplier: Multiplier for the outlier bounds (default: 3).
    """
    print("Loading data...")
    data = pd.read_csv(file_path)
    print(f"Original data shape: {data.shape}")
    print(f"Column names: {list(data.columns)}")
    
    # Check whether L(m) is present
    if 'L(m)' not in data.columns:
        raise ValueError("The data must contain an 'L(m)' column")
    
    # Handle missing values
    print(f"\nMissing values by column:\n{data.isnull().sum()}")
    initial_shape = data.shape
    data = data.dropna()
    print(f"Dropped {initial_shape[0] - data.shape[0]} rows with missing values")
    print(f"Data shape after dropping missing values: {data.shape}")
    
    # Keep a copy for comparison
    data_before_outlier = data.copy()
    
    # Detect outliers
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    print(f"\nOutlier detection with an IQR multiplier of {iqr_multiplier}:")
    print("-" * 50)
    
    if outlier_method == 'remove':
        # ========== Drop rows with outliers ==========
        # Create a Boolean mask for rows within the bounds
        mask = pd.Series([True] * len(data))
        outlier_details = {}
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # Identify outliers
            col_outliers_low = data[col] < lower_bound
            col_outliers_high = data[col] > upper_bound
            col_outliers = col_outliers_low | col_outliers_high
            
            # Count outliers
            n_outliers_low = col_outliers_low.sum()
            n_outliers_high = col_outliers_high.sum()
            n_outliers_total = col_outliers.sum()
            
            if n_outliers_total > 0:
                outlier_details[col] = {
                    'low': n_outliers_low,
                    'high': n_outliers_high,
                    'total': n_outliers_total,
                    'lower_bound': lower_bound,
                    'upper_bound': upper_bound
                }
                
                print(f"\nColumn '{col}':")
                print(f"  Range: [{data[col].min():.2f}, {data[col].max():.2f}]")
                print(f"  Outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
                print(f"  Outlier count: {n_outliers_total} ({n_outliers_total/len(data)*100:.1f}%)")
                print(f"    - Below the lower bound: {n_outliers_low}")
                print(f"    - Above the upper bound: {n_outliers_high}")
            
            # Keep rows within the bounds for every feature
            mask = mask & ~col_outliers
        
        # Apply the mask to remove outliers
        data_cleaned = data[mask].copy()
        n_removed = len(data) - len(data_cleaned)
        
        print(f"\n{'='*50}")
        print(f"Outlier removal summary:")
        print(f"  Original row count: {len(data)}")
        print(f"  Rows removed: {n_removed} ({n_removed/len(data)*100:.1f}%)")
        print(f"  Rows remaining: {len(data_cleaned)}")
        print(f"{'='*50}")
        
        data = data_cleaned
        
    else:
        # ========== Clip outliers ==========
        print("\nClipping outliers...")
        modified_count = 0
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # Count outliers
            outliers_low = (data[col] < lower_bound).sum()
            outliers_high = (data[col] > upper_bound).sum()
            
            if outliers_low > 0 or outliers_high > 0:
                print(f"\nColumn '{col}':")
                print(f"  - Values below the lower bound: {outliers_low}")
                print(f"  - Values above the upper bound: {outliers_high}")
                print(f"  - Bounds: [{lower_bound:.4f}, {upper_bound:.4f}]")
                modified_count += outliers_low + outliers_high
            
            # Clip outliers
            data[col] = data[col].clip(lower_bound, upper_bound)
        
        print(f"\nClipped {modified_count} outliers")
    
    print(f"\nFinal data shape: {data.shape}")
    
    # Compare statistics before and after preprocessing
    print("\n" + "="*50)
    print("Summary statistics comparison:")
    print("="*50)
    print("\nBefore preprocessing:")
    print(data_before_outlier.describe().round(2))
    print("\nAfter preprocessing:")
    print(data.describe().round(2))
    
    # Separate predictors from the target
    X = data.drop('L(m)', axis=1).values
    y = data['L(m)'].values.reshape(-1, 1)
    
    return X, y, data.columns.tolist()

# ==================== 2. Enhanced GAN architecture ====================
class FeatureAttention(nn.Module):
    """Feature attention mechanism."""
    def __init__(self, in_dim):
        super(FeatureAttention, self).__init__()
        self.fc1 = nn.Linear(in_dim, in_dim // 4)
        self.fc2 = nn.Linear(in_dim // 4, in_dim)
        self.sigmoid = nn.Sigmoid()
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        attn = self.fc1(x)
        attn = torch.relu(attn)
        attn = self.fc2(attn)
        attn = self.sigmoid(attn)
        attended = x * attn
        out = self.gamma * attended + x
        return out

class ResidualBlock(nn.Module):
    """Residual block."""
    def __init__(self, dim, dropout_rate=0.1):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim)
        )
        self.activation = nn.GELU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class EnhancedGenerator(nn.Module):
    """Enhanced generator network."""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], output_dim=1, dropout_rate=0.1):
        super(EnhancedGenerator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # Input projection
        layers.extend([
            nn.Linear(prev_dim, hidden_dims[0]),
            nn.LayerNorm(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        ])
        
        # Hidden layers
        for i in range(len(hidden_dims) - 1):
            layers.extend([
                nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                nn.LayerNorm(hidden_dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
            
            if i % 2 == 0:
                layers.append(ResidualBlock(hidden_dims[i+1], dropout_rate))
            
            if i == len(hidden_dims) // 2:
                layers.append(FeatureAttention(hidden_dims[i+1]))
        
        # Output layer
        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

class EnhancedDiscriminator(nn.Module):
    """Enhanced discriminator network."""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], dropout_rate=0.1):
        super(EnhancedDiscriminator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            if i < 2:
                layers.extend([
                    nn.utils.spectral_norm(nn.Linear(prev_dim, hidden_dim)),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            else:
                layers.extend([
                    nn.Linear(prev_dim, hidden_dim),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

# ==================== 3. Optimized GAN training ====================
class OptimizedGANRegressor:
    """Optimized GAN regression model."""
    def __init__(self, input_dim, device='cpu', weight_decay=5e-5):
        self.device = device
        self.input_dim = input_dim
        
        # Initialize the networks
        self.generator = EnhancedGenerator(input_dim).to(device)
        self.discriminator = EnhancedDiscriminator(input_dim + 1).to(device)
        
        # Optimizer
        self.g_optimizer = optim.AdamW(
            self.generator.parameters(), 
            lr=0.0002,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        self.d_optimizer = optim.AdamW(
            self.discriminator.parameters(), 
            lr=0.0001,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        
        # Learning-rate schedulers
        self.g_scheduler = CosineAnnealingWarmRestarts(self.g_optimizer, T_0=50, T_mult=2)
        self.d_scheduler = CosineAnnealingWarmRestarts(self.d_optimizer, T_0=50, T_mult=2)
        
        # Loss functions
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.regression_loss = nn.L1Loss()
        self.auxiliary_loss = nn.MSELoss()
        
        # Training history
        self.history = {
            'g_loss': [], 'd_loss': [], 'reg_loss': [],
            'train_r2': [], 'val_r2': []
        }
        
        # Early-stopping settings
        self.best_val_r2 = -float('inf')
        self.best_train_r2 = -float('inf')
        self.patience_counter = 0
        self.best_state = None
    
    def gradient_penalty(self, real_data, fake_data, lambda_gp=5):
        """Calculate the gradient penalty."""
        batch_size = real_data.size(0)
        alpha = torch.rand(batch_size, 1).to(self.device)
        alpha = alpha.expand_as(real_data)
        
        interpolated = alpha * real_data + (1 - alpha) * fake_data
        interpolated.requires_grad_(True)
        
        prob_interpolated = self.discriminator(interpolated)
        
        gradients = torch.autograd.grad(
            outputs=prob_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones_like(prob_interpolated),
            create_graph=True,
            retain_graph=True
        )[0]
        
        gradients = gradients.view(batch_size, -1)
        gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
        
        return gradient_penalty
    
    def train_step(self, real_x, real_y, epoch):
        """Run one training step."""
        batch_size = real_x.size(0)
        
        # Adjust label smoothing dynamically
        smooth_factor = max(0.8, 0.95 - epoch * 0.001)
        valid = torch.ones(batch_size, 1).to(self.device) * smooth_factor
        fake = torch.ones(batch_size, 1).to(self.device) * (1 - smooth_factor)
        
        # Train the discriminator
        self.d_optimizer.zero_grad()
        
        real_pairs = torch.cat([real_x, real_y], dim=1)
        real_pred = self.discriminator(real_pairs)
        d_real_loss = self.adversarial_loss(real_pred, valid)
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y.detach()], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        d_fake_loss = self.adversarial_loss(fake_pred, fake)
        
        gp = self.gradient_penalty(real_pairs, fake_pairs, lambda_gp=5)
        
        d_loss = (d_real_loss + d_fake_loss) / 2 + gp
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), 1.0)
        self.d_optimizer.step()
        
        # Train the generator
        self.g_optimizer.zero_grad()
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        
        g_adv_loss = self.adversarial_loss(fake_pred, valid)
        g_reg_loss_l1 = self.regression_loss(gen_y, real_y)
        g_reg_loss_mse = self.auxiliary_loss(gen_y, real_y)
        
        reg_weight = max(20, 30 - epoch * 0.05)
        
        g_loss = g_adv_loss + reg_weight * g_reg_loss_l1 + reg_weight * 0.5 * g_reg_loss_mse
        
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), 1.0)
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item(), g_reg_loss_l1.item()
    
    def fit(self, X_train, y_train, X_val, y_val, epochs=500, batch_size=32, patience=50, verbose=True):
        """Train the model."""
        self.generator.train()
        self.discriminator.train()
        
        # Create the data loader
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train), 
            torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        # Training loop
        for epoch in range(epochs):
            epoch_d_loss = []
            epoch_g_loss = []
            epoch_reg_loss = []
            
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                
                d_loss, g_loss, reg_loss = self.train_step(batch_x, batch_y, epoch)
                
                epoch_d_loss.append(d_loss)
                epoch_g_loss.append(g_loss)
                epoch_reg_loss.append(reg_loss)
            
            # Learning-rate schedule
            self.g_scheduler.step()
            self.d_scheduler.step()
            
            # Evaluate
            train_r2 = self.evaluate(X_train, y_train)
            val_r2 = self.evaluate(X_val, y_val)
            
            # Record training history
            self.history['d_loss'].append(np.mean(epoch_d_loss))
            self.history['g_loss'].append(np.mean(epoch_g_loss))
            self.history['reg_loss'].append(np.mean(epoch_reg_loss))
            self.history['train_r2'].append(train_r2)
            self.history['val_r2'].append(val_r2)
            
            # Save the best model
            combined_score = 0.4 * train_r2 + 0.6 * val_r2
            if combined_score > 0.4 * self.best_train_r2 + 0.6 * self.best_val_r2:
                self.best_val_r2 = val_r2
                self.best_train_r2 = train_r2
                self.patience_counter = 0
                self.best_state = {
                    'generator': self.generator.state_dict(),
                    'discriminator': self.discriminator.state_dict()
                }
            else:
                self.patience_counter += 1
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] "
                      f"D_loss: {np.mean(epoch_d_loss):.4f}, "
                      f"G_loss: {np.mean(epoch_g_loss):.4f}, "
                      f"Reg_loss: {np.mean(epoch_reg_loss):.4f}, "
                      f"Train_R2: {train_r2:.4f}, "
                      f"Val_R2: {val_r2:.4f}, "
                      f"Best_Train_R2: {self.best_train_r2:.4f}, "
                      f"Best_Val_R2: {self.best_val_r2:.4f}")
            
            # Early stopping
            if self.patience_counter >= patience and epoch > 100:
                print(f"\nEarly stopping triggered")
                print(f"Best training R²: {self.best_train_r2:.4f}")
                print(f"Best validation R²: {self.best_val_r2:.4f}")
                if self.best_state is not None:
                    self.generator.load_state_dict(self.best_state['generator'])
                    self.discriminator.load_state_dict(self.best_state['discriminator'])
                break
            
            # Continue training if training R² is still low
            if epoch == epochs - 1 and train_r2 < 0.6:
                print(f"\nTraining R² is still low ({train_r2:.4f}). Continuing training...")
                epochs = min(epochs + 100, 800)
    
    def predict(self, X):
        """Make predictions."""
        self.generator.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.generator(X_tensor).cpu().numpy()
        # Keep predictions positive
        predictions = np.maximum(predictions, 0)
        return predictions
    
    def evaluate(self, X, y):
        """Evaluate the model."""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 4. Ensemble wrapper ====================
class OptimizedEnsembleGAN:
    """Optimized GAN ensemble."""
    def __init__(self, n_models=3, input_dim=None, device='cpu'):
        self.n_models = n_models
        self.models = []
        self.input_dim = input_dim
        self.device = device
        self.weights = None
    
    def fit(self, X_train, y_train, X_val, y_val, **kwargs):
        """Train multiple models."""
        val_scores = []
        train_scores = []
        
        for i in range(self.n_models):
            print(f"\n{'='*50}")
            print(f"Training model {i+1}/{self.n_models}")
            print(f"{'='*50}")
            
            # Use a different seed for each model
            torch.manual_seed(42 + i * 10)
            np.random.seed(42 + i * 10)
            
            # Create and train the model
            model = OptimizedGANRegressor(
                input_dim=self.input_dim,
                device=self.device,
                weight_decay=5e-5 * (1 + i * 0.2)
            )
            
            # Train the model
            model.fit(X_train, y_train, X_val, y_val, **kwargs)
            
            # Evaluate
            train_score = model.evaluate(X_train, y_train)
            val_score = model.evaluate(X_val, y_val)
            
            train_scores.append(train_score)
            val_scores.append(val_score)
            self.models.append(model)
            
            print(f"\nResults for model {i+1}:")
            print(f"  Training R²: {train_score:.4f}")
            print(f"  Validation R²: {val_score:.4f}")
        
        # Calculate the averaging weights
        train_scores = np.array(train_scores)
        val_scores = np.array(val_scores)
        combined_scores = 0.4 * train_scores + 0.6 * val_scores
        
        self.weights = np.exp(combined_scores * 2) / np.sum(np.exp(combined_scores * 2))
        print(f"\nModel weights: {self.weights}")
    
    def predict(self, X):
        """Predict using the ensemble."""
        predictions = []
        for model in self.models:
            pred = model.predict(X)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        weighted_pred = np.average(predictions, axis=0, weights=self.weights)
        
        # Keep predictions positive
        weighted_pred = np.maximum(weighted_pred, 0)
        return weighted_pred
    
    def evaluate(self, X, y):
        """Evaluate the ensemble."""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 5. Model diagnostics and plots ====================
def diagnose_model(model, X_train, y_train, X_val, y_val, scaler_y):
    """Report model diagnostics."""
    print("\n" + "="*50)
    print("Model diagnostics")
    print("="*50)
    
    # Training performance
    y_train_pred = model.predict(X_train)
    y_train_pred_original = scaler_y.inverse_transform(y_train_pred)
    y_train_original = scaler_y.inverse_transform(y_train)
    
    train_r2 = r2_score(y_train_original, y_train_pred_original)
    train_mse = mean_squared_error(y_train_original, y_train_pred_original)
    train_mae = mean_absolute_error(y_train_original, y_train_pred_original)
    
    print(f"\nTraining performance (original scale):")
    print(f"  R² Score: {train_r2:.4f}")
    print(f"  MSE: {train_mse:.4f}")
    print(f"  MAE: {train_mae:.4f}")
    print(f"  RMSE: {np.sqrt(train_mse):.4f}")
    
    # Validation performance
    y_val_pred = model.predict(X_val)
    y_val_pred_original = scaler_y.inverse_transform(y_val_pred)
    y_val_original = scaler_y.inverse_transform(y_val)
    
    val_r2 = r2_score(y_val_original, y_val_pred_original)
    val_mse = mean_squared_error(y_val_original, y_val_pred_original)
    val_mae = mean_absolute_error(y_val_original, y_val_pred_original)
    
    print(f"\nValidation performance (original scale):")
    print(f"  R² Score: {val_r2:.4f}")
    print(f"  MSE: {val_mse:.4f}")
    print(f"  MAE: {val_mae:.4f}")
    print(f"  RMSE: {np.sqrt(val_mse):.4f}")
    
    # Summarize the diagnostics
    print("\n" + "-"*50)
    print("Diagnostic summary:")
    
    r2_diff = abs(train_r2 - val_r2)
    
    if train_r2 < 0.6:
        if val_r2 > 0.8:
            print("⚠️ Low training performance and good validation performance")
            print("  The training set may contain more noise or outliers")
        else:
            print("⚠️ Possible underfitting")
            print(f"  - Low training R²: {train_r2:.4f}")
    elif r2_diff > 0.15:
        print("⚠️ Possible overfitting")
        print(f"  - Training–validation R² gap: {r2_diff:.4f}")
    else:
        print("✅ Good model performance")
        print(f"  - Training R²: {train_r2:.4f}")
        print(f"  - Validation R²: {val_r2:.4f}")
        print(f"  - Difference: {r2_diff:.4f}")
    
    # Check whether the model meets the targets
    if val_r2 >= 0.8 and train_r2 >= 0.6:
        print("\n🎉 Targets met!")
        print(f"  - Validation R² >= 0.8: {val_r2:.4f}")
        print(f"  - Training R² >= 0.6: {train_r2:.4f}")
    elif val_r2 >= 0.8:
        print(f"\n⚠️ Validation met its target (R²={val_r2:.4f}). Training needs improvement (R²={train_r2:.4f}).")
    else:
        print(f"\n❌ Validation target missed (R²={val_r2:.4f} < 0.8)")
    
    return {
        'train_r2': train_r2, 'val_r2': val_r2,
        'train_mse': train_mse, 'val_mse': val_mse,
        'y_train_pred': y_train_pred_original,
        'y_val_pred': y_val_pred_original
    }

def plot_results(history, results, y_train_original, y_val_original):
    """Plot the results."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Read the training history
    if isinstance(history, list):
        plot_history = history[0]
    elif hasattr(history, 'models') and history.models:
        plot_history = history.models[0].history
    else:
        plot_history = history.history if hasattr(history, 'history') else history
    
    # 1. Loss curves
    ax = axes[0, 0]
    ax.plot(plot_history['d_loss'], label='Discriminator Loss', alpha=0.7)
    ax.plot(plot_history['g_loss'], label='Generator Loss', alpha=0.7)
    ax.plot(plot_history['reg_loss'], label='Regression Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training losses')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. R² curves
    ax = axes[0, 1]
    ax.plot(plot_history['train_r2'], label='Train R²', marker='o', markersize=2)
    ax.plot(plot_history['val_r2'], label='Validation R²', marker='s', markersize=2)
    ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target R²=0.8')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Min Train R²=0.6')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('R² Score')
    ax.set_title('R² over epochs')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. Training predictions vs. observations
    ax = axes[0, 2]
    ax.scatter(y_train_original, results['y_train_pred'], alpha=0.5, s=10)
    ax.plot([y_train_original.min(), y_train_original.max()], 
            [y_train_original.min(), y_train_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('Observed runout distance (m)')
    ax.set_ylabel('Predicted runout distance (m)')
    ax.set_title(f'Training set: R² = {results["train_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 4. Validation predictions vs. observations
    ax = axes[1, 0]
    ax.scatter(y_val_original, results['y_val_pred'], alpha=0.5, s=10, color='orange')
    ax.plot([y_val_original.min(), y_val_original.max()], 
            [y_val_original.min(), y_val_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('Observed runout distance (m)')
    ax.set_ylabel('Predicted runout distance (m)')
    ax.set_title(f'Validation set: R² = {results["val_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 5. Training residual distribution
    ax = axes[1, 1]
    residuals_train = y_train_original.flatten() - results['y_train_pred'].flatten()
    ax.hist(residuals_train, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('Residual (m)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Training residuals (mean={np.mean(residuals_train):.2f})')
    ax.grid(True, alpha=0.3)
    
    # 6. Validation residual distribution
    ax = axes[1, 2]
    residuals_val = y_val_original.flatten() - results['y_val_pred'].flatten()
    ax.hist(residuals_val, bins=30, edgecolor='black', alpha=0.7, color='orange')
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('Residual (m)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Validation residuals (mean={np.mean(residuals_val):.2f})')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('gan_outlier_removal_results.png', dpi=100, bbox_inches='tight')
    plt.show()

# ==================== 6. Main function ====================
def main(outlier_method='remove', iqr_multiplier=3, use_ensemble=True):
    """
    Run the runout distance prediction workflow.
    
    Parameters:
    - outlier_method: Choose 'remove' to drop outliers or 'clip' to cap them.
    - iqr_multiplier: Multiplier for the outlier bounds.
    - use_ensemble: Whether to train an ensemble.
    """
    # Select the computation device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    print(f"Outlier treatment: {outlier_method}")
    print(f"IQR multiplier: {iqr_multiplier}")
    print(f"Use ensemble: {use_ensemble}")
    print("="*50)
    
    # 1. Load the data and treat outliers
    X, y, column_names = load_and_preprocess_data(
        "imputed_others_with_runout.csv",
        outlier_method=outlier_method,
        iqr_multiplier=iqr_multiplier
    )
    
    # 2. Scale the data
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)
    
    # 3. Split the training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.2, random_state=42
    )
    
    print(f"\nData split:")
    print(f"Training set shape: {X_train.shape}")
    print(f"Validation set shape: {X_val.shape}")
    print(f"Feature count: {X_train.shape[1]}")
    
    # 4. Select the model type
    if use_ensemble:
        print("\nTraining the optimized GAN ensemble...")
        model = OptimizedEnsembleGAN(
            n_models=3, 
            input_dim=X_train.shape[1],
            device=device
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    else:
        print("\nTraining a single optimized GAN...")
        model = OptimizedGANRegressor(
            input_dim=X_train.shape[1],
            device=device,
            weight_decay=5e-5
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    
    # 5. Diagnose model performance
    results = diagnose_model(
        model, 
        X_train, y_train, 
        X_val, y_val,
        scaler_y
    )
    
    # 6. Plot the results
    y_train_original = scaler_y.inverse_transform(y_train)
    y_val_original = scaler_y.inverse_transform(y_val)
    plot_results(model, results, y_train_original, y_val_original)
    
    # 7. Report the final results
    print("\n" + "="*50)
    print("Final results")
    print("="*50)
    print(f"Outlier treatment: {outlier_method}")
    print(f"Rows after preprocessing: {X.shape[0]}")
    print(f"\nR² scores on the original scale:")
    print(f"  Training: {results['train_r2']:.4f}")
    print(f"  Validation: {results['val_r2']:.4f}")
    print(f"  Difference: {abs(results['train_r2'] - results['val_r2']):.4f}")
    # 8. Export predictions to CSV
    train_df = pd.DataFrame({
        "Dataset": "Train",
        "Actual": y_train_original.flatten(),
        "Predicted": results['y_train_pred'].flatten()
    })

    val_df = pd.DataFrame({
        "Dataset": "Validation",
        "Actual": y_val_original.flatten(),
        "Predicted": results['y_val_pred'].flatten()
    })

    output_df = pd.concat([train_df, val_df], ignore_index=True)
    output_df.to_csv("train_val_predictions.csv", index=False, encoding="utf-8-sig")
    print("\n✅ Predictions saved to train_val_predictions.csv")
    
    if results['val_r2'] >= 0.8 and results['train_r2'] >= 0.6:
        print("\n🎉 Both performance targets met!")
        print(f"  Validation R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  Training R² = {results['train_r2']:.4f} >= 0.6 ✓")
        print("  The model meets both configured R² targets")
    elif results['val_r2'] >= 0.8:
        print(f"\n⚠️ Only the validation target was met")
        print(f"  Validation R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  Training R² = {results['train_r2']:.4f} < 0.6 ✗")
        print("\nSuggestions:")
        print("  1. Check the quality of the training data")
        print("  2. Consider data cleaning or feature engineering")
        print("  3. Adjust the model architecture or training strategy")
    else:
        print(f"\n❌ Validation target missed")
        print(f"  Validation R² = {results['val_r2']:.4f} < 0.8")
        print("\nSuggestions:")
        print("  1. Increase model capacity")
        print("  2. Collect more training data")
        print("  3. Try a different model architecture")
    
    return model, results

# Run the workflow
if __name__ == "__main__":
    # Adjustable parameters:
    # outlier_method: 'remove' to drop outliers or 'clip' to cap them
    # iqr_multiplier: 1.5 (strict), 3 (standard), or 5 (lenient)
    # use_ensemble: True for an ensemble or False for a single model
    
    model, results = main(
        outlier_method='clip',  # Clip outliers
        iqr_multiplier=1.5,         # Use an IQR multiplier of 1.5
        use_ensemble=True         # Use the ensemble
    )
    
    print("\nTraining complete. Figure saved as 'gan_outlier_removal_results.png'.")

In [ ]:
#Rainfall
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# ==================== 1. Data loading and preprocessing ====================
def load_and_preprocess_data(file_path, outlier_method='remove', iqr_multiplier=3):
    """
    Load and preprocess the data.
    
    Parameters:
    - file_path: Path to the input data.
    - outlier_method: Use 'remove' to drop outliers or 'clip' to cap them.
    - iqr_multiplier: Multiplier for the outlier bounds (default: 3).
    """
    print("Loading data...")
    data = pd.read_csv(file_path)
    print(f"Original data shape: {data.shape}")
    print(f"Column names: {list(data.columns)}")
    
    # Check whether L(m) is present
    if 'L(m)' not in data.columns:
        raise ValueError("The data must contain an 'L(m)' column")
    
    # Handle missing values
    print(f"\nMissing values by column:\n{data.isnull().sum()}")
    initial_shape = data.shape
    data = data.dropna()
    print(f"Dropped {initial_shape[0] - data.shape[0]} rows with missing values")
    print(f"Data shape after dropping missing values: {data.shape}")
    
    # Keep a copy for comparison
    data_before_outlier = data.copy()
    
    # Detect outliers
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    print(f"\nOutlier detection with an IQR multiplier of {iqr_multiplier}:")
    print("-" * 50)
    
    if outlier_method == 'remove':
        # ========== Drop rows with outliers ==========
        # Create a Boolean mask for rows within the bounds
        mask = pd.Series([True] * len(data))
        outlier_details = {}
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # Identify outliers
            col_outliers_low = data[col] < lower_bound
            col_outliers_high = data[col] > upper_bound
            col_outliers = col_outliers_low | col_outliers_high
            
            # Count outliers
            n_outliers_low = col_outliers_low.sum()
            n_outliers_high = col_outliers_high.sum()
            n_outliers_total = col_outliers.sum()
            
            if n_outliers_total > 0:
                outlier_details[col] = {
                    'low': n_outliers_low,
                    'high': n_outliers_high,
                    'total': n_outliers_total,
                    'lower_bound': lower_bound,
                    'upper_bound': upper_bound
                }
                
                print(f"\nColumn '{col}':")
                print(f"  Range: [{data[col].min():.2f}, {data[col].max():.2f}]")
                print(f"  Outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
                print(f"  Outlier count: {n_outliers_total} ({n_outliers_total/len(data)*100:.1f}%)")
                print(f"    - Below the lower bound: {n_outliers_low}")
                print(f"    - Above the upper bound: {n_outliers_high}")
            
            # Keep rows within the bounds for every feature
            mask = mask & ~col_outliers
        
        # Apply the mask to remove outliers
        data_cleaned = data[mask].copy()
        n_removed = len(data) - len(data_cleaned)
        
        print(f"\n{'='*50}")
        print(f"Outlier removal summary:")
        print(f"  Original row count: {len(data)}")
        print(f"  Rows removed: {n_removed} ({n_removed/len(data)*100:.1f}%)")
        print(f"  Rows remaining: {len(data_cleaned)}")
        print(f"{'='*50}")
        
        data = data_cleaned
        
    else:
        # ========== Clip outliers ==========
        print("\nClipping outliers...")
        modified_count = 0
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # Count outliers
            outliers_low = (data[col] < lower_bound).sum()
            outliers_high = (data[col] > upper_bound).sum()
            
            if outliers_low > 0 or outliers_high > 0:
                print(f"\nColumn '{col}':")
                print(f"  - Values below the lower bound: {outliers_low}")
                print(f"  - Values above the upper bound: {outliers_high}")
                print(f"  - Bounds: [{lower_bound:.4f}, {upper_bound:.4f}]")
                modified_count += outliers_low + outliers_high
            
            # Clip outliers
            data[col] = data[col].clip(lower_bound, upper_bound)
        
        print(f"\nClipped {modified_count} outliers")
    
    print(f"\nFinal data shape: {data.shape}")
    
    # Compare statistics before and after preprocessing
    print("\n" + "="*50)
    print("Summary statistics comparison:")
    print("="*50)
    print("\nBefore preprocessing:")
    print(data_before_outlier.describe().round(2))
    print("\nAfter preprocessing:")
    print(data.describe().round(2))
    
    # Separate predictors from the target
    X = data.drop('L(m)', axis=1).values
    y = data['L(m)'].values.reshape(-1, 1)
    
    return X, y, data.columns.tolist()

# ==================== 2. Enhanced GAN architecture ====================
class FeatureAttention(nn.Module):
    """Feature attention mechanism."""
    def __init__(self, in_dim):
        super(FeatureAttention, self).__init__()
        self.fc1 = nn.Linear(in_dim, in_dim // 4)
        self.fc2 = nn.Linear(in_dim // 4, in_dim)
        self.sigmoid = nn.Sigmoid()
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        attn = self.fc1(x)
        attn = torch.relu(attn)
        attn = self.fc2(attn)
        attn = self.sigmoid(attn)
        attended = x * attn
        out = self.gamma * attended + x
        return out

class ResidualBlock(nn.Module):
    """Residual block."""
    def __init__(self, dim, dropout_rate=0.1):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim)
        )
        self.activation = nn.GELU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class EnhancedGenerator(nn.Module):
    """Enhanced generator network."""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], output_dim=1, dropout_rate=0.1):
        super(EnhancedGenerator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # Input projection
        layers.extend([
            nn.Linear(prev_dim, hidden_dims[0]),
            nn.LayerNorm(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        ])
        
        # Hidden layers
        for i in range(len(hidden_dims) - 1):
            layers.extend([
                nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                nn.LayerNorm(hidden_dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
            
            if i % 2 == 0:
                layers.append(ResidualBlock(hidden_dims[i+1], dropout_rate))
            
            if i == len(hidden_dims) // 2:
                layers.append(FeatureAttention(hidden_dims[i+1]))
        
        # Output layer
        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

class EnhancedDiscriminator(nn.Module):
    """Enhanced discriminator network."""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], dropout_rate=0.1):
        super(EnhancedDiscriminator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            if i < 2:
                layers.extend([
                    nn.utils.spectral_norm(nn.Linear(prev_dim, hidden_dim)),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            else:
                layers.extend([
                    nn.Linear(prev_dim, hidden_dim),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

# ==================== 3. Optimized GAN training ====================
class OptimizedGANRegressor:
    """Optimized GAN regression model."""
    def __init__(self, input_dim, device='cpu', weight_decay=5e-5):
        self.device = device
        self.input_dim = input_dim
        
        # Initialize the networks
        self.generator = EnhancedGenerator(input_dim).to(device)
        self.discriminator = EnhancedDiscriminator(input_dim + 1).to(device)
        
        # Optimizer
        self.g_optimizer = optim.AdamW(
            self.generator.parameters(), 
            lr=0.0002,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        self.d_optimizer = optim.AdamW(
            self.discriminator.parameters(), 
            lr=0.0001,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        
        # Learning-rate schedulers
        self.g_scheduler = CosineAnnealingWarmRestarts(self.g_optimizer, T_0=50, T_mult=2)
        self.d_scheduler = CosineAnnealingWarmRestarts(self.d_optimizer, T_0=50, T_mult=2)
        
        # Loss functions
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.regression_loss = nn.L1Loss()
        self.auxiliary_loss = nn.MSELoss()
        
        # Training history
        self.history = {
            'g_loss': [], 'd_loss': [], 'reg_loss': [],
            'train_r2': [], 'val_r2': []
        }
        
        # Early-stopping settings
        self.best_val_r2 = -float('inf')
        self.best_train_r2 = -float('inf')
        self.patience_counter = 0
        self.best_state = None
    
    def gradient_penalty(self, real_data, fake_data, lambda_gp=5):
        """Calculate the gradient penalty."""
        batch_size = real_data.size(0)
        alpha = torch.rand(batch_size, 1).to(self.device)
        alpha = alpha.expand_as(real_data)
        
        interpolated = alpha * real_data + (1 - alpha) * fake_data
        interpolated.requires_grad_(True)
        
        prob_interpolated = self.discriminator(interpolated)
        
        gradients = torch.autograd.grad(
            outputs=prob_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones_like(prob_interpolated),
            create_graph=True,
            retain_graph=True
        )[0]
        
        gradients = gradients.view(batch_size, -1)
        gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
        
        return gradient_penalty
    
    def train_step(self, real_x, real_y, epoch):
        """Run one training step."""
        batch_size = real_x.size(0)
        
        # Adjust label smoothing dynamically
        smooth_factor = max(0.8, 0.95 - epoch * 0.001)
        valid = torch.ones(batch_size, 1).to(self.device) * smooth_factor
        fake = torch.ones(batch_size, 1).to(self.device) * (1 - smooth_factor)
        
        # Train the discriminator
        self.d_optimizer.zero_grad()
        
        real_pairs = torch.cat([real_x, real_y], dim=1)
        real_pred = self.discriminator(real_pairs)
        d_real_loss = self.adversarial_loss(real_pred, valid)
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y.detach()], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        d_fake_loss = self.adversarial_loss(fake_pred, fake)
        
        gp = self.gradient_penalty(real_pairs, fake_pairs, lambda_gp=5)
        
        d_loss = (d_real_loss + d_fake_loss) / 2 + gp
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), 1.0)
        self.d_optimizer.step()
        
        # Train the generator
        self.g_optimizer.zero_grad()
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        
        g_adv_loss = self.adversarial_loss(fake_pred, valid)
        g_reg_loss_l1 = self.regression_loss(gen_y, real_y)
        g_reg_loss_mse = self.auxiliary_loss(gen_y, real_y)
        
        reg_weight = max(20, 30 - epoch * 0.05)
        
        g_loss = g_adv_loss + reg_weight * g_reg_loss_l1 + reg_weight * 0.5 * g_reg_loss_mse
        
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), 1.0)
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item(), g_reg_loss_l1.item()
    
    def fit(self, X_train, y_train, X_val, y_val, epochs=500, batch_size=32, patience=50, verbose=True):
        """Train the model."""
        self.generator.train()
        self.discriminator.train()
        
        # Create the data loader
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train), 
            torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        # Training loop
        for epoch in range(epochs):
            epoch_d_loss = []
            epoch_g_loss = []
            epoch_reg_loss = []
            
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                
                d_loss, g_loss, reg_loss = self.train_step(batch_x, batch_y, epoch)
                
                epoch_d_loss.append(d_loss)
                epoch_g_loss.append(g_loss)
                epoch_reg_loss.append(reg_loss)
            
            # Learning-rate schedule
            self.g_scheduler.step()
            self.d_scheduler.step()
            
            # Evaluate
            train_r2 = self.evaluate(X_train, y_train)
            val_r2 = self.evaluate(X_val, y_val)
            
            # Record training history
            self.history['d_loss'].append(np.mean(epoch_d_loss))
            self.history['g_loss'].append(np.mean(epoch_g_loss))
            self.history['reg_loss'].append(np.mean(epoch_reg_loss))
            self.history['train_r2'].append(train_r2)
            self.history['val_r2'].append(val_r2)
            
            # Save the best model
            combined_score = 0.4 * train_r2 + 0.6 * val_r2
            if combined_score > 0.4 * self.best_train_r2 + 0.6 * self.best_val_r2:
                self.best_val_r2 = val_r2
                self.best_train_r2 = train_r2
                self.patience_counter = 0
                self.best_state = {
                    'generator': self.generator.state_dict(),
                    'discriminator': self.discriminator.state_dict()
                }
            else:
                self.patience_counter += 1
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] "
                      f"D_loss: {np.mean(epoch_d_loss):.4f}, "
                      f"G_loss: {np.mean(epoch_g_loss):.4f}, "
                      f"Reg_loss: {np.mean(epoch_reg_loss):.4f}, "
                      f"Train_R2: {train_r2:.4f}, "
                      f"Val_R2: {val_r2:.4f}, "
                      f"Best_Train_R2: {self.best_train_r2:.4f}, "
                      f"Best_Val_R2: {self.best_val_r2:.4f}")
            
            # Early stopping
            if self.patience_counter >= patience and epoch > 100:
                print(f"\nEarly stopping triggered")
                print(f"Best training R²: {self.best_train_r2:.4f}")
                print(f"Best validation R²: {self.best_val_r2:.4f}")
                if self.best_state is not None:
                    self.generator.load_state_dict(self.best_state['generator'])
                    self.discriminator.load_state_dict(self.best_state['discriminator'])
                break
            
            # Continue training if training R² is still low
            if epoch == epochs - 1 and train_r2 < 0.6:
                print(f"\nTraining R² is still low ({train_r2:.4f}). Continuing training...")
                epochs = min(epochs + 100, 800)
    
    def predict(self, X):
        """Make predictions."""
        self.generator.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.generator(X_tensor).cpu().numpy()
        return predictions
    
    def evaluate(self, X, y):
        """Evaluate the model."""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 4. Ensemble wrapper ====================
class OptimizedEnsembleGAN:
    """Optimized GAN ensemble."""
    def __init__(self, n_models=3, input_dim=None, device='cpu'):
        self.n_models = n_models
        self.models = []
        self.input_dim = input_dim
        self.device = device
        self.weights = None
    
    def fit(self, X_train, y_train, X_val, y_val, **kwargs):
        """Train multiple models."""
        val_scores = []
        train_scores = []
        
        for i in range(self.n_models):
            print(f"\n{'='*50}")
            print(f"Training model {i+1}/{self.n_models}")
            print(f"{'='*50}")
            
            # Use a different seed for each model
            torch.manual_seed(42 + i * 10)
            np.random.seed(42 + i * 10)
            
            # Create and train the model
            model = OptimizedGANRegressor(
                input_dim=self.input_dim,
                device=self.device,
                weight_decay=5e-5 * (1 + i * 0.2)
            )
            
            # Train the model
            model.fit(X_train, y_train, X_val, y_val, **kwargs)
            
            # Evaluate
            train_score = model.evaluate(X_train, y_train)
            val_score = model.evaluate(X_val, y_val)
            
            train_scores.append(train_score)
            val_scores.append(val_score)
            self.models.append(model)
            
            print(f"\nResults for model {i+1}:")
            print(f"  Training R²: {train_score:.4f}")
            print(f"  Validation R²: {val_score:.4f}")
        
        # Calculate the averaging weights
        train_scores = np.array(train_scores)
        val_scores = np.array(val_scores)
        combined_scores = 0.4 * train_scores + 0.6 * val_scores
        
        self.weights = np.exp(combined_scores * 2) / np.sum(np.exp(combined_scores * 2))
        print(f"\nModel weights: {self.weights}")
    
    def predict(self, X):
        """Predict using the ensemble."""
        predictions = []
        for model in self.models:
            pred = model.predict(X)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        weighted_pred = np.average(predictions, axis=0, weights=self.weights)
        
        return weighted_pred
    
    def evaluate(self, X, y):
        """Evaluate the ensemble."""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 5. Model diagnostics and plots ====================
def diagnose_model(model, X_train, y_train, X_val, y_val, scaler_y):
    """Report model diagnostics."""
    print("\n" + "="*50)
    print("Model diagnostics")
    print("="*50)
    
    # Training performance
    y_train_pred = model.predict(X_train)
    y_train_pred_original = scaler_y.inverse_transform(y_train_pred)
    y_train_original = scaler_y.inverse_transform(y_train)
    
    train_r2 = r2_score(y_train_original, y_train_pred_original)
    train_mse = mean_squared_error(y_train_original, y_train_pred_original)
    train_mae = mean_absolute_error(y_train_original, y_train_pred_original)
    
    print(f"\nTraining performance (original scale):")
    print(f"  R² Score: {train_r2:.4f}")
    print(f"  MSE: {train_mse:.4f}")
    print(f"  MAE: {train_mae:.4f}")
    print(f"  RMSE: {np.sqrt(train_mse):.4f}")
    
    # Validation performance
    y_val_pred = model.predict(X_val)
    y_val_pred_original = scaler_y.inverse_transform(y_val_pred)
    y_val_original = scaler_y.inverse_transform(y_val)
    
    val_r2 = r2_score(y_val_original, y_val_pred_original)
    val_mse = mean_squared_error(y_val_original, y_val_pred_original)
    val_mae = mean_absolute_error(y_val_original, y_val_pred_original)
    
    print(f"\nValidation performance (original scale):")
    print(f"  R² Score: {val_r2:.4f}")
    print(f"  MSE: {val_mse:.4f}")
    print(f"  MAE: {val_mae:.4f}")
    print(f"  RMSE: {np.sqrt(val_mse):.4f}")
    
    # Summarize the diagnostics
    print("\n" + "-"*50)
    print("Diagnostic summary:")
    
    r2_diff = abs(train_r2 - val_r2)
    
    if train_r2 < 0.6:
        if val_r2 > 0.8:
            print("⚠️ Low training performance and good validation performance")
            print("  The training set may contain more noise or outliers")
        else:
            print("⚠️ Possible underfitting")
            print(f"  - Low training R²: {train_r2:.4f}")
    elif r2_diff > 0.15:
        print("⚠️ Possible overfitting")
        print(f"  - Training–validation R² gap: {r2_diff:.4f}")
    else:
        print("✅ Good model performance")
        print(f"  - Training R²: {train_r2:.4f}")
        print(f"  - Validation R²: {val_r2:.4f}")
        print(f"  - Difference: {r2_diff:.4f}")
    
    # Check whether the model meets the targets
    if val_r2 >= 0.8 and train_r2 >= 0.6:
        print("\n🎉 Targets met!")
        print(f"  - Validation R² >= 0.8: {val_r2:.4f}")
        print(f"  - Training R² >= 0.6: {train_r2:.4f}")
    elif val_r2 >= 0.8:
        print(f"\n⚠️ Validation met its target (R²={val_r2:.4f}). Training needs improvement (R²={train_r2:.4f}).")
    else:
        print(f"\n❌ Validation target missed (R²={val_r2:.4f} < 0.8)")
    
    return {
        'train_r2': train_r2, 'val_r2': val_r2,
        'train_mse': train_mse, 'val_mse': val_mse,
        'y_train_pred': y_train_pred_original,
        'y_val_pred': y_val_pred_original
    }

def plot_results(history, results, y_train_original, y_val_original):
    """Plot the results."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Read the training history
    if isinstance(history, list):
        plot_history = history[0]
    elif hasattr(history, 'models') and history.models:
        plot_history = history.models[0].history
    else:
        plot_history = history.history if hasattr(history, 'history') else history
    
    # 1. Loss curves
    ax = axes[0, 0]
    ax.plot(plot_history['d_loss'], label='Discriminator Loss', alpha=0.7)
    ax.plot(plot_history['g_loss'], label='Generator Loss', alpha=0.7)
    ax.plot(plot_history['reg_loss'], label='Regression Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training losses')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. R² curves
    ax = axes[0, 1]
    ax.plot(plot_history['train_r2'], label='Train R²', marker='o', markersize=2)
    ax.plot(plot_history['val_r2'], label='Validation R²', marker='s', markersize=2)
    ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target R²=0.8')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Min Train R²=0.6')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('R² Score')
    ax.set_title('R² over epochs')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. Training predictions vs. observations
    ax = axes[0, 2]
    ax.scatter(y_train_original, results['y_train_pred'], alpha=0.5, s=10)
    ax.plot([y_train_original.min(), y_train_original.max()], 
            [y_train_original.min(), y_train_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('Observed runout distance (m)')
    ax.set_ylabel('Predicted runout distance (m)')
    ax.set_title(f'Training set: R² = {results["train_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 4. Validation predictions vs. observations
    ax = axes[1, 0]
    ax.scatter(y_val_original, results['y_val_pred'], alpha=0.5, s=10, color='orange')
    ax.plot([y_val_original.min(), y_val_original.max()], 
            [y_val_original.min(), y_val_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('Observed runout distance (m)')
    ax.set_ylabel('Predicted runout distance (m)')
    ax.set_title(f'Validation set: R² = {results["val_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 5. Training residual distribution
    ax = axes[1, 1]
    residuals_train = y_train_original.flatten() - results['y_train_pred'].flatten()
    ax.hist(residuals_train, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('Residual (m)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Training residuals (mean={np.mean(residuals_train):.2f})')
    ax.grid(True, alpha=0.3)
    
    # 6. Validation residual distribution
    ax = axes[1, 2]
    residuals_val = y_val_original.flatten() - results['y_val_pred'].flatten()
    ax.hist(residuals_val, bins=30, edgecolor='black', alpha=0.7, color='orange')
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('Residual (m)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Validation residuals (mean={np.mean(residuals_val):.2f})')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('gan_outlier_removal_results.png', dpi=100, bbox_inches='tight')
    plt.show()

# ==================== 6. Main function ====================
def main(outlier_method='remove', iqr_multiplier=3, use_ensemble=True):
    """
    Run the runout distance prediction workflow.
    
    Parameters:
    - outlier_method: Choose 'remove' to drop outliers or 'clip' to cap them.
    - iqr_multiplier: Multiplier for the outlier bounds.
    - use_ensemble: Whether to train an ensemble.
    """
    # Select the computation device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    print(f"Outlier treatment: {outlier_method}")
    print(f"IQR multiplier: {iqr_multiplier}")
    print(f"Use ensemble: {use_ensemble}")
    print("="*50)
    
    # 1. Load the data and treat outliers
    X, y, column_names = load_and_preprocess_data(
        "imputed_rainfall_with_runout.csv",
        outlier_method=outlier_method,
        iqr_multiplier=iqr_multiplier
    )
    
    # 2. Scale the data
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)
    
    # 3. Split the training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.2, random_state=42
    )
    
    print(f"\nData split:")
    print(f"Training set shape: {X_train.shape}")
    print(f"Validation set shape: {X_val.shape}")
    print(f"Feature count: {X_train.shape[1]}")
    
    # 4. Select the model type
    if use_ensemble:
        print("\nTraining the optimized GAN ensemble...")
        model = OptimizedEnsembleGAN(
            n_models=3, 
            input_dim=X_train.shape[1],
            device=device
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    else:
        print("\nTraining a single optimized GAN...")
        model = OptimizedGANRegressor(
            input_dim=X_train.shape[1],
            device=device,
            weight_decay=5e-5
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    
    # 5. Diagnose model performance
    results = diagnose_model(
        model, 
        X_train, y_train, 
        X_val, y_val,
        scaler_y
    )
    
    # 6. Plot the results
    y_train_original = scaler_y.inverse_transform(y_train)
    y_val_original = scaler_y.inverse_transform(y_val)
    plot_results(model, results, y_train_original, y_val_original)
    
    # 7. Report the final results
    print("\n" + "="*50)
    print("Final results")
    print("="*50)
    print(f"Outlier treatment: {outlier_method}")
    print(f"Rows after preprocessing: {X.shape[0]}")
    print(f"\nR² scores on the original scale:")
    print(f"  Training: {results['train_r2']:.4f}")
    print(f"  Validation: {results['val_r2']:.4f}")
    print(f"  Difference: {abs(results['train_r2'] - results['val_r2']):.4f}")
    # 8. Export predictions to CSV
    train_df = pd.DataFrame({
        "Dataset": "Train",
        "Actual": y_train_original.flatten(),
        "Predicted": results['y_train_pred'].flatten()
    })

    val_df = pd.DataFrame({
        "Dataset": "Validation",
        "Actual": y_val_original.flatten(),
        "Predicted": results['y_val_pred'].flatten()
    })

    output_df = pd.concat([train_df, val_df], ignore_index=True)
    output_df.to_csv("train_val_predictions.csv", index=False, encoding="utf-8-sig")
    print("\n✅ Predictions saved to train_val_predictions.csv")

    if results['val_r2'] >= 0.8 and results['train_r2'] >= 0.6:
        print("\n🎉 Both performance targets met!")
        print(f"  Validation R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  Training R² = {results['train_r2']:.4f} >= 0.6 ✓")
        print("  The model meets both configured R² targets")
    elif results['val_r2'] >= 0.8:
        print(f"\n⚠️ Only the validation target was met")
        print(f"  Validation R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  Training R² = {results['train_r2']:.4f} < 0.6 ✗")
        print("\nSuggestions:")
        print("  1. Check the quality of the training data")
        print("  2. Consider data cleaning or feature engineering")
        print("  3. Adjust the model architecture or training strategy")
    else:
        print(f"\n❌ Validation target missed")
        print(f"  Validation R² = {results['val_r2']:.4f} < 0.8")
        print("\nSuggestions:")
        print("  1. Increase model capacity")
        print("  2. Collect more training data")
        print("  3. Try a different model architecture")
    
    return model, results

# Run the workflow
if __name__ == "__main__":
    # Adjustable parameters:
    # outlier_method: 'remove' to drop outliers or 'clip' to cap them
    # iqr_multiplier: 1.5 (strict), 3 (standard), or 5 (lenient)
    # use_ensemble: True for an ensemble or False for a single model
    
    model, results = main(
        outlier_method='clip',  # Clip outliers
        iqr_multiplier=1.5,         # Use an IQR multiplier of 1.5
        use_ensemble=True         # Use the ensemble
    )
    
    print("\nTraining complete. Figure saved as 'gan_outlier_removal_results.png'.")

In [ ]:
#rock
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# ==================== 1. Data loading and preprocessing ====================
def load_and_preprocess_data(file_path, outlier_method='remove', iqr_multiplier=3):
    """
    Load and preprocess the data.
    
    Parameters:
    - file_path: Path to the input data.
    - outlier_method: Use 'remove' to drop outliers or 'clip' to cap them.
    - iqr_multiplier: Multiplier for the outlier bounds (default: 3).
    """
    print("Loading data...")
    data = pd.read_csv(file_path)
    print(f"Original data shape: {data.shape}")
    print(f"Column names: {list(data.columns)}")
    
    # Check whether L(m) is present
    if 'L(m)' not in data.columns:
        raise ValueError("The data must contain an 'L(m)' column")
    
    # Handle missing values
    print(f"\nMissing values by column:\n{data.isnull().sum()}")
    initial_shape = data.shape
    data = data.dropna()
    print(f"Dropped {initial_shape[0] - data.shape[0]} rows with missing values")
    print(f"Data shape after dropping missing values: {data.shape}")
    
    # Keep a copy for comparison
    data_before_outlier = data.copy()
    
    # Detect outliers
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    print(f"\nOutlier detection with an IQR multiplier of {iqr_multiplier}:")
    print("-" * 50)
    
    if outlier_method == 'remove':
        # ========== Drop rows with outliers ==========
        # Create a Boolean mask for rows within the bounds
        mask = pd.Series([True] * len(data))
        outlier_details = {}
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # Identify outliers
            col_outliers_low = data[col] < lower_bound
            col_outliers_high = data[col] > upper_bound
            col_outliers = col_outliers_low | col_outliers_high
            
            # Count outliers
            n_outliers_low = col_outliers_low.sum()
            n_outliers_high = col_outliers_high.sum()
            n_outliers_total = col_outliers.sum()
            
            if n_outliers_total > 0:
                outlier_details[col] = {
                    'low': n_outliers_low,
                    'high': n_outliers_high,
                    'total': n_outliers_total,
                    'lower_bound': lower_bound,
                    'upper_bound': upper_bound
                }
                
                print(f"\nColumn '{col}':")
                print(f"  Range: [{data[col].min():.2f}, {data[col].max():.2f}]")
                print(f"  Outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
                print(f"  Outlier count: {n_outliers_total} ({n_outliers_total/len(data)*100:.1f}%)")
                print(f"    - Below the lower bound: {n_outliers_low}")
                print(f"    - Above the upper bound: {n_outliers_high}")
            
            # Keep rows within the bounds for every feature
            mask = mask & ~col_outliers
        
        # Apply the mask to remove outliers
        data_cleaned = data[mask].copy()
        n_removed = len(data) - len(data_cleaned)
        
        print(f"\n{'='*50}")
        print(f"Outlier removal summary:")
        print(f"  Original row count: {len(data)}")
        print(f"  Rows removed: {n_removed} ({n_removed/len(data)*100:.1f}%)")
        print(f"  Rows remaining: {len(data_cleaned)}")
        print(f"{'='*50}")
        
        data = data_cleaned
        
    else:
        # ========== Clip outliers ==========
        print("\nClipping outliers...")
        modified_count = 0
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # Count outliers
            outliers_low = (data[col] < lower_bound).sum()
            outliers_high = (data[col] > upper_bound).sum()
            
            if outliers_low > 0 or outliers_high > 0:
                print(f"\nColumn '{col}':")
                print(f"  - Values below the lower bound: {outliers_low}")
                print(f"  - Values above the upper bound: {outliers_high}")
                print(f"  - Bounds: [{lower_bound:.4f}, {upper_bound:.4f}]")
                modified_count += outliers_low + outliers_high
            
            # Clip outliers
            data[col] = data[col].clip(lower_bound, upper_bound)
        
        print(f"\nClipped {modified_count} outliers")
    
    print(f"\nFinal data shape: {data.shape}")
    
    # Compare statistics before and after preprocessing
    print("\n" + "="*50)
    print("Summary statistics comparison:")
    print("="*50)
    print("\nBefore preprocessing:")
    print(data_before_outlier.describe().round(2))
    print("\nAfter preprocessing:")
    print(data.describe().round(2))
    
    # Separate predictors from the target
    X = data.drop('L(m)', axis=1).values
    y = data['L(m)'].values.reshape(-1, 1)
    
    return X, y, data.columns.tolist()

# ==================== 2. Enhanced GAN architecture ====================
class FeatureAttention(nn.Module):
    """Feature attention mechanism."""
    def __init__(self, in_dim):
        super(FeatureAttention, self).__init__()
        self.fc1 = nn.Linear(in_dim, in_dim // 4)
        self.fc2 = nn.Linear(in_dim // 4, in_dim)
        self.sigmoid = nn.Sigmoid()
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        attn = self.fc1(x)
        attn = torch.relu(attn)
        attn = self.fc2(attn)
        attn = self.sigmoid(attn)
        attended = x * attn
        out = self.gamma * attended + x
        return out

class ResidualBlock(nn.Module):
    """Residual block."""
    def __init__(self, dim, dropout_rate=0.1):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim)
        )
        self.activation = nn.GELU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class EnhancedGenerator(nn.Module):
    """Enhanced generator network."""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], output_dim=1, dropout_rate=0.1):
        super(EnhancedGenerator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # Input projection
        layers.extend([
            nn.Linear(prev_dim, hidden_dims[0]),
            nn.LayerNorm(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        ])
        
        # Hidden layers
        for i in range(len(hidden_dims) - 1):
            layers.extend([
                nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                nn.LayerNorm(hidden_dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
            
            if i % 2 == 0:
                layers.append(ResidualBlock(hidden_dims[i+1], dropout_rate))
            
            if i == len(hidden_dims) // 2:
                layers.append(FeatureAttention(hidden_dims[i+1]))
        
        # Output layer
        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

class EnhancedDiscriminator(nn.Module):
    """Enhanced discriminator network."""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], dropout_rate=0.1):
        super(EnhancedDiscriminator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            if i < 2:
                layers.extend([
                    nn.utils.spectral_norm(nn.Linear(prev_dim, hidden_dim)),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            else:
                layers.extend([
                    nn.Linear(prev_dim, hidden_dim),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

# ==================== 3. Optimized GAN training ====================
class OptimizedGANRegressor:
    """Optimized GAN regression model."""
    def __init__(self, input_dim, device='cpu', weight_decay=5e-5):
        self.device = device
        self.input_dim = input_dim
        
        # Initialize the networks
        self.generator = EnhancedGenerator(input_dim).to(device)
        self.discriminator = EnhancedDiscriminator(input_dim + 1).to(device)
        
        # Optimizer
        self.g_optimizer = optim.AdamW(
            self.generator.parameters(), 
            lr=0.0002,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        self.d_optimizer = optim.AdamW(
            self.discriminator.parameters(), 
            lr=0.0001,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        
        # Learning-rate schedulers
        self.g_scheduler = CosineAnnealingWarmRestarts(self.g_optimizer, T_0=50, T_mult=2)
        self.d_scheduler = CosineAnnealingWarmRestarts(self.d_optimizer, T_0=50, T_mult=2)
        
        # Loss functions
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.regression_loss = nn.L1Loss()
        self.auxiliary_loss = nn.MSELoss()
        
        # Training history
        self.history = {
            'g_loss': [], 'd_loss': [], 'reg_loss': [],
            'train_r2': [], 'val_r2': []
        }
        
        # Early-stopping settings
        self.best_val_r2 = -float('inf')
        self.best_train_r2 = -float('inf')
        self.patience_counter = 0
        self.best_state = None
    
    def gradient_penalty(self, real_data, fake_data, lambda_gp=5):
        """Calculate the gradient penalty."""
        batch_size = real_data.size(0)
        alpha = torch.rand(batch_size, 1).to(self.device)
        alpha = alpha.expand_as(real_data)
        
        interpolated = alpha * real_data + (1 - alpha) * fake_data
        interpolated.requires_grad_(True)
        
        prob_interpolated = self.discriminator(interpolated)
        
        gradients = torch.autograd.grad(
            outputs=prob_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones_like(prob_interpolated),
            create_graph=True,
            retain_graph=True
        )[0]
        
        gradients = gradients.view(batch_size, -1)
        gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
        
        return gradient_penalty
    
    def train_step(self, real_x, real_y, epoch):
        """Run one training step."""
        batch_size = real_x.size(0)
        
        # Adjust label smoothing dynamically
        smooth_factor = max(0.8, 0.95 - epoch * 0.001)
        valid = torch.ones(batch_size, 1).to(self.device) * smooth_factor
        fake = torch.ones(batch_size, 1).to(self.device) * (1 - smooth_factor)
        
        # Train the discriminator
        self.d_optimizer.zero_grad()
        
        real_pairs = torch.cat([real_x, real_y], dim=1)
        real_pred = self.discriminator(real_pairs)
        d_real_loss = self.adversarial_loss(real_pred, valid)
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y.detach()], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        d_fake_loss = self.adversarial_loss(fake_pred, fake)
        
        gp = self.gradient_penalty(real_pairs, fake_pairs, lambda_gp=5)
        
        d_loss = (d_real_loss + d_fake_loss) / 2 + gp
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), 1.0)
        self.d_optimizer.step()
        
        # Train the generator
        self.g_optimizer.zero_grad()
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        
        g_adv_loss = self.adversarial_loss(fake_pred, valid)
        g_reg_loss_l1 = self.regression_loss(gen_y, real_y)
        g_reg_loss_mse = self.auxiliary_loss(gen_y, real_y)
        
        reg_weight = max(20, 30 - epoch * 0.05)
        
        g_loss = g_adv_loss + reg_weight * g_reg_loss_l1 + reg_weight * 0.5 * g_reg_loss_mse
        
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), 1.0)
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item(), g_reg_loss_l1.item()
    
    def fit(self, X_train, y_train, X_val, y_val, epochs=500, batch_size=32, patience=50, verbose=True):
        """Train the model."""
        self.generator.train()
        self.discriminator.train()
        
        # Create the data loader
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train), 
            torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        # Training loop
        for epoch in range(epochs):
            epoch_d_loss = []
            epoch_g_loss = []
            epoch_reg_loss = []
            
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                
                d_loss, g_loss, reg_loss = self.train_step(batch_x, batch_y, epoch)
                
                epoch_d_loss.append(d_loss)
                epoch_g_loss.append(g_loss)
                epoch_reg_loss.append(reg_loss)
            
            # Learning-rate schedule
            self.g_scheduler.step()
            self.d_scheduler.step()
            
            # Evaluate
            train_r2 = self.evaluate(X_train, y_train)
            val_r2 = self.evaluate(X_val, y_val)
            
            # Record training history
            self.history['d_loss'].append(np.mean(epoch_d_loss))
            self.history['g_loss'].append(np.mean(epoch_g_loss))
            self.history['reg_loss'].append(np.mean(epoch_reg_loss))
            self.history['train_r2'].append(train_r2)
            self.history['val_r2'].append(val_r2)
            
            # Save the best model
            combined_score = 0.4 * train_r2 + 0.6 * val_r2
            if combined_score > 0.4 * self.best_train_r2 + 0.6 * self.best_val_r2:
                self.best_val_r2 = val_r2
                self.best_train_r2 = train_r2
                self.patience_counter = 0
                self.best_state = {
                    'generator': self.generator.state_dict(),
                    'discriminator': self.discriminator.state_dict()
                }
            else:
                self.patience_counter += 1
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] "
                      f"D_loss: {np.mean(epoch_d_loss):.4f}, "
                      f"G_loss: {np.mean(epoch_g_loss):.4f}, "
                      f"Reg_loss: {np.mean(epoch_reg_loss):.4f}, "
                      f"Train_R2: {train_r2:.4f}, "
                      f"Val_R2: {val_r2:.4f}, "
                      f"Best_Train_R2: {self.best_train_r2:.4f}, "
                      f"Best_Val_R2: {self.best_val_r2:.4f}")
            
            # Early stopping
            if self.patience_counter >= patience and epoch > 100:
                print(f"\nEarly stopping triggered")
                print(f"Best training R²: {self.best_train_r2:.4f}")
                print(f"Best validation R²: {self.best_val_r2:.4f}")
                if self.best_state is not None:
                    self.generator.load_state_dict(self.best_state['generator'])
                    self.discriminator.load_state_dict(self.best_state['discriminator'])
                break
            
            # Continue training if training R² is still low
            if epoch == epochs - 1 and train_r2 < 0.6:
                print(f"\nTraining R² is still low ({train_r2:.4f}). Continuing training...")
                epochs = min(epochs + 100, 800)
    
    def predict(self, X):
        """Make predictions."""
        self.generator.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.generator(X_tensor).cpu().numpy()
        return predictions
    
    def evaluate(self, X, y):
        """Evaluate the model."""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 4. Ensemble wrapper ====================
class OptimizedEnsembleGAN:
    """Optimized GAN ensemble."""
    def __init__(self, n_models=3, input_dim=None, device='cpu'):
        self.n_models = n_models
        self.models = []
        self.input_dim = input_dim
        self.device = device
        self.weights = None
    
    def fit(self, X_train, y_train, X_val, y_val, **kwargs):
        """Train multiple models."""
        val_scores = []
        train_scores = []
        
        for i in range(self.n_models):
            print(f"\n{'='*50}")
            print(f"Training model {i+1}/{self.n_models}")
            print(f"{'='*50}")
            
            # Use a different seed for each model
            torch.manual_seed(42 + i * 10)
            np.random.seed(42 + i * 10)
            
            # Create and train the model
            model = OptimizedGANRegressor(
                input_dim=self.input_dim,
                device=self.device,
                weight_decay=5e-5 * (1 + i * 0.2)
            )
            
            # Train the model
            model.fit(X_train, y_train, X_val, y_val, **kwargs)
            
            # Evaluate
            train_score = model.evaluate(X_train, y_train)
            val_score = model.evaluate(X_val, y_val)
            
            train_scores.append(train_score)
            val_scores.append(val_score)
            self.models.append(model)
            
            print(f"\nResults for model {i+1}:")
            print(f"  Training R²: {train_score:.4f}")
            print(f"  Validation R²: {val_score:.4f}")
        
        # Calculate the averaging weights
        train_scores = np.array(train_scores)
        val_scores = np.array(val_scores)
        combined_scores = 0.4 * train_scores + 0.6 * val_scores
        
        self.weights = np.exp(combined_scores * 2) / np.sum(np.exp(combined_scores * 2))
        print(f"\nModel weights: {self.weights}")
    
    def predict(self, X):
        """Predict using the ensemble."""
        predictions = []
        for model in self.models:
            pred = model.predict(X)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        weighted_pred = np.average(predictions, axis=0, weights=self.weights)
        
        return weighted_pred
    
    def evaluate(self, X, y):
        """Evaluate the ensemble."""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 5. Model diagnostics and plots ====================
def diagnose_model(model, X_train, y_train, X_val, y_val, scaler_y):
    """Report model diagnostics."""
    print("\n" + "="*50)
    print("Model diagnostics")
    print("="*50)
    
    # Training performance
    y_train_pred = model.predict(X_train)
    y_train_pred_original = scaler_y.inverse_transform(y_train_pred)
    y_train_original = scaler_y.inverse_transform(y_train)
    
    train_r2 = r2_score(y_train_original, y_train_pred_original)
    train_mse = mean_squared_error(y_train_original, y_train_pred_original)
    train_mae = mean_absolute_error(y_train_original, y_train_pred_original)
    
    print(f"\nTraining performance (original scale):")
    print(f"  R² Score: {train_r2:.4f}")
    print(f"  MSE: {train_mse:.4f}")
    print(f"  MAE: {train_mae:.4f}")
    print(f"  RMSE: {np.sqrt(train_mse):.4f}")
    
    # Validation performance
    y_val_pred = model.predict(X_val)
    y_val_pred_original = scaler_y.inverse_transform(y_val_pred)
    y_val_original = scaler_y.inverse_transform(y_val)
    
    val_r2 = r2_score(y_val_original, y_val_pred_original)
    val_mse = mean_squared_error(y_val_original, y_val_pred_original)
    val_mae = mean_absolute_error(y_val_original, y_val_pred_original)
    
    print(f"\nValidation performance (original scale):")
    print(f"  R² Score: {val_r2:.4f}")
    print(f"  MSE: {val_mse:.4f}")
    print(f"  MAE: {val_mae:.4f}")
    print(f"  RMSE: {np.sqrt(val_mse):.4f}")
    
    # Summarize the diagnostics
    print("\n" + "-"*50)
    print("Diagnostic summary:")
    
    r2_diff = abs(train_r2 - val_r2)
    
    if train_r2 < 0.6:
        if val_r2 > 0.8:
            print("⚠️ Low training performance and good validation performance")
            print("  The training set may contain more noise or outliers")
        else:
            print("⚠️ Possible underfitting")
            print(f"  - Low training R²: {train_r2:.4f}")
    elif r2_diff > 0.15:
        print("⚠️ Possible overfitting")
        print(f"  - Training–validation R² gap: {r2_diff:.4f}")
    else:
        print("✅ Good model performance")
        print(f"  - Training R²: {train_r2:.4f}")
        print(f"  - Validation R²: {val_r2:.4f}")
        print(f"  - Difference: {r2_diff:.4f}")
    
    # Check whether the model meets the targets
    if val_r2 >= 0.8 and train_r2 >= 0.6:
        print("\n🎉 Targets met!")
        print(f"  - Validation R² >= 0.8: {val_r2:.4f}")
        print(f"  - Training R² >= 0.6: {train_r2:.4f}")
    elif val_r2 >= 0.8:
        print(f"\n⚠️ Validation met its target (R²={val_r2:.4f}). Training needs improvement (R²={train_r2:.4f}).")
    else:
        print(f"\n❌ Validation target missed (R²={val_r2:.4f} < 0.8)")
    
    return {
        'train_r2': train_r2, 'val_r2': val_r2,
        'train_mse': train_mse, 'val_mse': val_mse,
        'y_train_pred': y_train_pred_original,
        'y_val_pred': y_val_pred_original
    }

def plot_results(history, results, y_train_original, y_val_original):
    """Plot the results."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Read the training history
    if isinstance(history, list):
        plot_history = history[0]
    elif hasattr(history, 'models') and history.models:
        plot_history = history.models[0].history
    else:
        plot_history = history.history if hasattr(history, 'history') else history
    
    # 1. Loss curves
    ax = axes[0, 0]
    ax.plot(plot_history['d_loss'], label='Discriminator Loss', alpha=0.7)
    ax.plot(plot_history['g_loss'], label='Generator Loss', alpha=0.7)
    ax.plot(plot_history['reg_loss'], label='Regression Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training losses')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. R² curves
    ax = axes[0, 1]
    ax.plot(plot_history['train_r2'], label='Train R²', marker='o', markersize=2)
    ax.plot(plot_history['val_r2'], label='Validation R²', marker='s', markersize=2)
    ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target R²=0.8')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Min Train R²=0.6')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('R² Score')
    ax.set_title('R² over epochs')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. Training predictions vs. observations
    ax = axes[0, 2]
    ax.scatter(y_train_original, results['y_train_pred'], alpha=0.5, s=10)
    ax.plot([y_train_original.min(), y_train_original.max()], 
            [y_train_original.min(), y_train_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('Observed runout distance (m)')
    ax.set_ylabel('Predicted runout distance (m)')
    ax.set_title(f'Training set: R² = {results["train_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 4. Validation predictions vs. observations
    ax = axes[1, 0]
    ax.scatter(y_val_original, results['y_val_pred'], alpha=0.5, s=10, color='orange')
    ax.plot([y_val_original.min(), y_val_original.max()], 
            [y_val_original.min(), y_val_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('Observed runout distance (m)')
    ax.set_ylabel('Predicted runout distance (m)')
    ax.set_title(f'Validation set: R² = {results["val_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 5. Training residual distribution
    ax = axes[1, 1]
    residuals_train = y_train_original.flatten() - results['y_train_pred'].flatten()
    ax.hist(residuals_train, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('Residual (m)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Training residuals (mean={np.mean(residuals_train):.2f})')
    ax.grid(True, alpha=0.3)
    
    # 6. Validation residual distribution
    ax = axes[1, 2]
    residuals_val = y_val_original.flatten() - results['y_val_pred'].flatten()
    ax.hist(residuals_val, bins=30, edgecolor='black', alpha=0.7, color='orange')
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('Residual (m)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Validation residuals (mean={np.mean(residuals_val):.2f})')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('gan_outlier_removal_results.png', dpi=100, bbox_inches='tight')
    plt.show()

# ==================== 6. Main function ====================
def main(outlier_method='remove', iqr_multiplier=3, use_ensemble=True):
    """
    Run the runout distance prediction workflow.
    
    Parameters:
    - outlier_method: Choose 'remove' to drop outliers or 'clip' to cap them.
    - iqr_multiplier: Multiplier for the outlier bounds.
    - use_ensemble: Whether to train an ensemble.
    """
    # Select the computation device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    print(f"Outlier treatment: {outlier_method}")
    print(f"IQR multiplier: {iqr_multiplier}")
    print(f"Use ensemble: {use_ensemble}")
    print("="*50)
    
    # 1. Load the data and treat outliers
    X, y, column_names = load_and_preprocess_data(
        "imputed_rock_with_runout.csv",
        outlier_method=outlier_method,
        iqr_multiplier=iqr_multiplier
    )
    
    # 2. Scale the data
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)
    
    # 3. Split the training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.2, random_state=42
    )
    
    print(f"\nData split:")
    print(f"Training set shape: {X_train.shape}")
    print(f"Validation set shape: {X_val.shape}")
    print(f"Feature count: {X_train.shape[1]}")
    
    # 4. Select the model type
    if use_ensemble:
        print("\nTraining the optimized GAN ensemble...")
        model = OptimizedEnsembleGAN(
            n_models=3, 
            input_dim=X_train.shape[1],
            device=device
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    else:
        print("\nTraining a single optimized GAN...")
        model = OptimizedGANRegressor(
            input_dim=X_train.shape[1],
            device=device,
            weight_decay=5e-5
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    
    # 5. Diagnose model performance
    results = diagnose_model(
        model, 
        X_train, y_train, 
        X_val, y_val,
        scaler_y
    )
    
    # 6. Plot the results
    y_train_original = scaler_y.inverse_transform(y_train)
    y_val_original = scaler_y.inverse_transform(y_val)
    plot_results(model, results, y_train_original, y_val_original)
    
    # 7. Report the final results
    print("\n" + "="*50)
    print("Final results")
    print("="*50)
    print(f"Outlier treatment: {outlier_method}")
    print(f"Rows after preprocessing: {X.shape[0]}")
    print(f"\nR² scores on the original scale:")
    print(f"  Training: {results['train_r2']:.4f}")
    print(f"  Validation: {results['val_r2']:.4f}")
    print(f"  Difference: {abs(results['train_r2'] - results['val_r2']):.4f}")
    # 8. Export predictions to CSV
    train_df = pd.DataFrame({
        "Dataset": "Train",
        "Actual": y_train_original.flatten(),
        "Predicted": results['y_train_pred'].flatten()
    })

    val_df = pd.DataFrame({
        "Dataset": "Validation",
        "Actual": y_val_original.flatten(),
        "Predicted": results['y_val_pred'].flatten()
    })

    output_df = pd.concat([train_df, val_df], ignore_index=True)
    output_df.to_csv("train_val_predictions.csv", index=False, encoding="utf-8-sig")
    print("\n✅ Predictions saved to train_val_predictions.csv")
    
    if results['val_r2'] >= 0.8 and results['train_r2'] >= 0.6:
        print("\n🎉 Both performance targets met!")
        print(f"  Validation R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  Training R² = {results['train_r2']:.4f} >= 0.6 ✓")
        print("  The model meets both configured R² targets")
    elif results['val_r2'] >= 0.8:
        print(f"\n⚠️ Only the validation target was met")
        print(f"  Validation R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  Training R² = {results['train_r2']:.4f} < 0.6 ✗")
        print("\nSuggestions:")
        print("  1. Check the quality of the training data")
        print("  2. Consider data cleaning or feature engineering")
        print("  3. Adjust the model architecture or training strategy")
    else:
        print(f"\n❌ Validation target missed")
        print(f"  Validation R² = {results['val_r2']:.4f} < 0.8")
        print("\nSuggestions:")
        print("  1. Increase model capacity")
        print("  2. Collect more training data")
        print("  3. Try a different model architecture")
    
    return model, results

# Run the workflow
if __name__ == "__main__":
    # Adjustable parameters:
    # outlier_method: 'remove' to drop outliers or 'clip' to cap them
    # iqr_multiplier: 1.5 (strict), 3 (standard), or 5 (lenient)
    # use_ensemble: True for an ensemble or False for a single model
    
    model, results = main(
        outlier_method='clip',  # Clip outliers
        iqr_multiplier=1.5,         # Use an IQR multiplier of 1.5
        use_ensemble=True         # Use the ensemble
    )
    
    print("\nTraining complete. Figure saved as 'gan_outlier_removal_results.png'.")

In [ ]:
#soil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# ==================== 1. Data loading and preprocessing ====================
def load_and_preprocess_data(file_path, outlier_method='remove', iqr_multiplier=3):
    """
    Load and preprocess the data.
    
    Parameters:
    - file_path: Path to the input data.
    - outlier_method: Use 'remove' to drop outliers or 'clip' to cap them.
    - iqr_multiplier: Multiplier for the outlier bounds (default: 3).
    """
    print("Loading data...")
    data = pd.read_csv(file_path)
    print(f"Original data shape: {data.shape}")
    print(f"Column names: {list(data.columns)}")
    
    # Check whether L(m) is present
    if 'L(m)' not in data.columns:
        raise ValueError("The data must contain an 'L(m)' column")
    
    # Handle missing values
    print(f"\nMissing values by column:\n{data.isnull().sum()}")
    initial_shape = data.shape
    data = data.dropna()
    print(f"Dropped {initial_shape[0] - data.shape[0]} rows with missing values")
    print(f"Data shape after dropping missing values: {data.shape}")
    
    # Keep a copy for comparison
    data_before_outlier = data.copy()
    
    # Detect outliers
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    print(f"\nOutlier detection with an IQR multiplier of {iqr_multiplier}:")
    print("-" * 50)
    
    if outlier_method == 'remove':
        # ========== Drop rows with outliers ==========
        # Create a Boolean mask for rows within the bounds
        mask = pd.Series([True] * len(data))
        outlier_details = {}
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # Identify outliers
            col_outliers_low = data[col] < lower_bound
            col_outliers_high = data[col] > upper_bound
            col_outliers = col_outliers_low | col_outliers_high
            
            # Count outliers
            n_outliers_low = col_outliers_low.sum()
            n_outliers_high = col_outliers_high.sum()
            n_outliers_total = col_outliers.sum()
            
            if n_outliers_total > 0:
                outlier_details[col] = {
                    'low': n_outliers_low,
                    'high': n_outliers_high,
                    'total': n_outliers_total,
                    'lower_bound': lower_bound,
                    'upper_bound': upper_bound
                }
                
                print(f"\nColumn '{col}':")
                print(f"  Range: [{data[col].min():.2f}, {data[col].max():.2f}]")
                print(f"  Outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
                print(f"  Outlier count: {n_outliers_total} ({n_outliers_total/len(data)*100:.1f}%)")
                print(f"    - Below the lower bound: {n_outliers_low}")
                print(f"    - Above the upper bound: {n_outliers_high}")
            
            # Keep rows within the bounds for every feature
            mask = mask & ~col_outliers
        
        # Apply the mask to remove outliers
        data_cleaned = data[mask].copy()
        n_removed = len(data) - len(data_cleaned)
        
        print(f"\n{'='*50}")
        print(f"Outlier removal summary:")
        print(f"  Original row count: {len(data)}")
        print(f"  Rows removed: {n_removed} ({n_removed/len(data)*100:.1f}%)")
        print(f"  Rows remaining: {len(data_cleaned)}")
        print(f"{'='*50}")
        
        data = data_cleaned
        
    else:
        # ========== Clip outliers ==========
        print("\nClipping outliers...")
        modified_count = 0
        
        for col in data.columns:
            lower_bound = Q1[col] - iqr_multiplier * IQR[col]
            upper_bound = Q3[col] + iqr_multiplier * IQR[col]
            
            # Count outliers
            outliers_low = (data[col] < lower_bound).sum()
            outliers_high = (data[col] > upper_bound).sum()
            
            if outliers_low > 0 or outliers_high > 0:
                print(f"\nColumn '{col}':")
                print(f"  - Values below the lower bound: {outliers_low}")
                print(f"  - Values above the upper bound: {outliers_high}")
                print(f"  - Bounds: [{lower_bound:.4f}, {upper_bound:.4f}]")
                modified_count += outliers_low + outliers_high
            
            # Clip outliers
            data[col] = data[col].clip(lower_bound, upper_bound)
        
        print(f"\nClipped {modified_count} outliers")
    
    print(f"\nFinal data shape: {data.shape}")
    
    # Compare statistics before and after preprocessing
    print("\n" + "="*50)
    print("Summary statistics comparison:")
    print("="*50)
    print("\nBefore preprocessing:")
    print(data_before_outlier.describe().round(2))
    print("\nAfter preprocessing:")
    print(data.describe().round(2))
    
    # Separate predictors from the target
    X = data.drop('L(m)', axis=1).values
    y = data['L(m)'].values.reshape(-1, 1)
    
    return X, y, data.columns.tolist()

# ==================== 2. Enhanced GAN architecture ====================
class FeatureAttention(nn.Module):
    """Feature attention mechanism."""
    def __init__(self, in_dim):
        super(FeatureAttention, self).__init__()
        self.fc1 = nn.Linear(in_dim, in_dim // 4)
        self.fc2 = nn.Linear(in_dim // 4, in_dim)
        self.sigmoid = nn.Sigmoid()
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        attn = self.fc1(x)
        attn = torch.relu(attn)
        attn = self.fc2(attn)
        attn = self.sigmoid(attn)
        attended = x * attn
        out = self.gamma * attended + x
        return out

class ResidualBlock(nn.Module):
    """Residual block."""
    def __init__(self, dim, dropout_rate=0.1):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim)
        )
        self.activation = nn.GELU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class EnhancedGenerator(nn.Module):
    """Enhanced generator network."""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], output_dim=1, dropout_rate=0.1):
        super(EnhancedGenerator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # Input projection
        layers.extend([
            nn.Linear(prev_dim, hidden_dims[0]),
            nn.LayerNorm(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        ])
        
        # Hidden layers
        for i in range(len(hidden_dims) - 1):
            layers.extend([
                nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                nn.LayerNorm(hidden_dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
            
            if i % 2 == 0:
                layers.append(ResidualBlock(hidden_dims[i+1], dropout_rate))
            
            if i == len(hidden_dims) // 2:
                layers.append(FeatureAttention(hidden_dims[i+1]))
        
        # Output layer
        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

class EnhancedDiscriminator(nn.Module):
    """Enhanced discriminator network."""
    def __init__(self, input_dim, hidden_dims=[128, 256, 512, 256, 128], dropout_rate=0.1):
        super(EnhancedDiscriminator, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            if i < 2:
                layers.extend([
                    nn.utils.spectral_norm(nn.Linear(prev_dim, hidden_dim)),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            else:
                layers.extend([
                    nn.Linear(prev_dim, hidden_dim),
                    nn.LeakyReLU(0.2),
                    nn.Dropout(dropout_rate)
                ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.model = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
    
    def forward(self, x):
        return self.model(x)

# ==================== 3. Optimized GAN training ====================
class OptimizedGANRegressor:
    """Optimized GAN regression model."""
    def __init__(self, input_dim, device='cpu', weight_decay=5e-5):
        self.device = device
        self.input_dim = input_dim
        
        # Initialize the networks
        self.generator = EnhancedGenerator(input_dim).to(device)
        self.discriminator = EnhancedDiscriminator(input_dim + 1).to(device)
        
        # Optimizer
        self.g_optimizer = optim.AdamW(
            self.generator.parameters(), 
            lr=0.0002,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        self.d_optimizer = optim.AdamW(
            self.discriminator.parameters(), 
            lr=0.0001,
            betas=(0.5, 0.999),
            weight_decay=weight_decay
        )
        
        # Learning-rate schedulers
        self.g_scheduler = CosineAnnealingWarmRestarts(self.g_optimizer, T_0=50, T_mult=2)
        self.d_scheduler = CosineAnnealingWarmRestarts(self.d_optimizer, T_0=50, T_mult=2)
        
        # Loss functions
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.regression_loss = nn.L1Loss()
        self.auxiliary_loss = nn.MSELoss()
        
        # Training history
        self.history = {
            'g_loss': [], 'd_loss': [], 'reg_loss': [],
            'train_r2': [], 'val_r2': []
        }
        
        # Early-stopping settings
        self.best_val_r2 = -float('inf')
        self.best_train_r2 = -float('inf')
        self.patience_counter = 0
        self.best_state = None
    
    def gradient_penalty(self, real_data, fake_data, lambda_gp=5):
        """Calculate the gradient penalty."""
        batch_size = real_data.size(0)
        alpha = torch.rand(batch_size, 1).to(self.device)
        alpha = alpha.expand_as(real_data)
        
        interpolated = alpha * real_data + (1 - alpha) * fake_data
        interpolated.requires_grad_(True)
        
        prob_interpolated = self.discriminator(interpolated)
        
        gradients = torch.autograd.grad(
            outputs=prob_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones_like(prob_interpolated),
            create_graph=True,
            retain_graph=True
        )[0]
        
        gradients = gradients.view(batch_size, -1)
        gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
        
        return gradient_penalty
    
    def train_step(self, real_x, real_y, epoch):
        """Run one training step."""
        batch_size = real_x.size(0)
        
        # Adjust label smoothing dynamically
        smooth_factor = max(0.8, 0.95 - epoch * 0.001)
        valid = torch.ones(batch_size, 1).to(self.device) * smooth_factor
        fake = torch.ones(batch_size, 1).to(self.device) * (1 - smooth_factor)
        
        # Train the discriminator
        self.d_optimizer.zero_grad()
        
        real_pairs = torch.cat([real_x, real_y], dim=1)
        real_pred = self.discriminator(real_pairs)
        d_real_loss = self.adversarial_loss(real_pred, valid)
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y.detach()], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        d_fake_loss = self.adversarial_loss(fake_pred, fake)
        
        gp = self.gradient_penalty(real_pairs, fake_pairs, lambda_gp=5)
        
        d_loss = (d_real_loss + d_fake_loss) / 2 + gp
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), 1.0)
        self.d_optimizer.step()
        
        # Train the generator
        self.g_optimizer.zero_grad()
        
        gen_y = self.generator(real_x)
        fake_pairs = torch.cat([real_x, gen_y], dim=1)
        fake_pred = self.discriminator(fake_pairs)
        
        g_adv_loss = self.adversarial_loss(fake_pred, valid)
        g_reg_loss_l1 = self.regression_loss(gen_y, real_y)
        g_reg_loss_mse = self.auxiliary_loss(gen_y, real_y)
        
        reg_weight = max(20, 30 - epoch * 0.05)
        
        g_loss = g_adv_loss + reg_weight * g_reg_loss_l1 + reg_weight * 0.5 * g_reg_loss_mse
        
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), 1.0)
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item(), g_reg_loss_l1.item()
    
    def fit(self, X_train, y_train, X_val, y_val, epochs=500, batch_size=32, patience=50, verbose=True):
        """Train the model."""
        self.generator.train()
        self.discriminator.train()
        
        # Create the data loader
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train), 
            torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        # Training loop
        for epoch in range(epochs):
            epoch_d_loss = []
            epoch_g_loss = []
            epoch_reg_loss = []
            
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                
                d_loss, g_loss, reg_loss = self.train_step(batch_x, batch_y, epoch)
                
                epoch_d_loss.append(d_loss)
                epoch_g_loss.append(g_loss)
                epoch_reg_loss.append(reg_loss)
            
            # Learning-rate schedule
            self.g_scheduler.step()
            self.d_scheduler.step()
            
            # Evaluate
            train_r2 = self.evaluate(X_train, y_train)
            val_r2 = self.evaluate(X_val, y_val)
            
            # Record training history
            self.history['d_loss'].append(np.mean(epoch_d_loss))
            self.history['g_loss'].append(np.mean(epoch_g_loss))
            self.history['reg_loss'].append(np.mean(epoch_reg_loss))
            self.history['train_r2'].append(train_r2)
            self.history['val_r2'].append(val_r2)
            
            # Save the best model
            combined_score = 0.4 * train_r2 + 0.6 * val_r2
            if combined_score > 0.4 * self.best_train_r2 + 0.6 * self.best_val_r2:
                self.best_val_r2 = val_r2
                self.best_train_r2 = train_r2
                self.patience_counter = 0
                self.best_state = {
                    'generator': self.generator.state_dict(),
                    'discriminator': self.discriminator.state_dict()
                }
            else:
                self.patience_counter += 1
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] "
                      f"D_loss: {np.mean(epoch_d_loss):.4f}, "
                      f"G_loss: {np.mean(epoch_g_loss):.4f}, "
                      f"Reg_loss: {np.mean(epoch_reg_loss):.4f}, "
                      f"Train_R2: {train_r2:.4f}, "
                      f"Val_R2: {val_r2:.4f}, "
                      f"Best_Train_R2: {self.best_train_r2:.4f}, "
                      f"Best_Val_R2: {self.best_val_r2:.4f}")
            
            # Early stopping
            if self.patience_counter >= patience and epoch > 100:
                print(f"\nEarly stopping triggered")
                print(f"Best training R²: {self.best_train_r2:.4f}")
                print(f"Best validation R²: {self.best_val_r2:.4f}")
                if self.best_state is not None:
                    self.generator.load_state_dict(self.best_state['generator'])
                    self.discriminator.load_state_dict(self.best_state['discriminator'])
                break
            
            # Continue training if training R² is still low
            if epoch == epochs - 1 and train_r2 < 0.6:
                print(f"\nTraining R² is still low ({train_r2:.4f}). Continuing training...")
                epochs = min(epochs + 100, 800)
    
    def predict(self, X):
        """Make predictions."""
        self.generator.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.generator(X_tensor).cpu().numpy()
        return predictions
    
    def evaluate(self, X, y):
        """Evaluate the model."""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 4. Ensemble wrapper ====================
class OptimizedEnsembleGAN:
    """Optimized GAN ensemble."""
    def __init__(self, n_models=3, input_dim=None, device='cpu'):
        self.n_models = n_models
        self.models = []
        self.input_dim = input_dim
        self.device = device
        self.weights = None
    
    def fit(self, X_train, y_train, X_val, y_val, **kwargs):
        """Train multiple models."""
        val_scores = []
        train_scores = []
        
        for i in range(self.n_models):
            print(f"\n{'='*50}")
            print(f"Training model {i+1}/{self.n_models}")
            print(f"{'='*50}")
            
            # Use a different seed for each model
            torch.manual_seed(42 + i * 10)
            np.random.seed(42 + i * 10)
            
            # Create and train the model
            model = OptimizedGANRegressor(
                input_dim=self.input_dim,
                device=self.device,
                weight_decay=5e-5 * (1 + i * 0.2)
            )
            
            # Train the model
            model.fit(X_train, y_train, X_val, y_val, **kwargs)
            
            # Evaluate
            train_score = model.evaluate(X_train, y_train)
            val_score = model.evaluate(X_val, y_val)
            
            train_scores.append(train_score)
            val_scores.append(val_score)
            self.models.append(model)
            
            print(f"\nResults for model {i+1}:")
            print(f"  Training R²: {train_score:.4f}")
            print(f"  Validation R²: {val_score:.4f}")
        
        # Calculate the averaging weights
        train_scores = np.array(train_scores)
        val_scores = np.array(val_scores)
        combined_scores = 0.4 * train_scores + 0.6 * val_scores
        
        self.weights = np.exp(combined_scores * 2) / np.sum(np.exp(combined_scores * 2))
        print(f"\nModel weights: {self.weights}")
    
    def predict(self, X):
        """Predict using the ensemble."""
        predictions = []
        for model in self.models:
            pred = model.predict(X)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        weighted_pred = np.average(predictions, axis=0, weights=self.weights)
        
        return weighted_pred
    
    def evaluate(self, X, y):
        """Evaluate the ensemble."""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

# ==================== 5. Model diagnostics and plots ====================
def diagnose_model(model, X_train, y_train, X_val, y_val, scaler_y):
    """Report model diagnostics."""
    print("\n" + "="*50)
    print("Model diagnostics")
    print("="*50)
    
    # Training performance
    y_train_pred = model.predict(X_train)
    y_train_pred_original = scaler_y.inverse_transform(y_train_pred)
    y_train_original = scaler_y.inverse_transform(y_train)
    
    train_r2 = r2_score(y_train_original, y_train_pred_original)
    train_mse = mean_squared_error(y_train_original, y_train_pred_original)
    train_mae = mean_absolute_error(y_train_original, y_train_pred_original)
    
    print(f"\nTraining performance (original scale):")
    print(f"  R² Score: {train_r2:.4f}")
    print(f"  MSE: {train_mse:.4f}")
    print(f"  MAE: {train_mae:.4f}")
    print(f"  RMSE: {np.sqrt(train_mse):.4f}")
    
    # Validation performance
    y_val_pred = model.predict(X_val)
    y_val_pred_original = scaler_y.inverse_transform(y_val_pred)
    y_val_original = scaler_y.inverse_transform(y_val)
    
    val_r2 = r2_score(y_val_original, y_val_pred_original)
    val_mse = mean_squared_error(y_val_original, y_val_pred_original)
    val_mae = mean_absolute_error(y_val_original, y_val_pred_original)
    
    print(f"\nValidation performance (original scale):")
    print(f"  R² Score: {val_r2:.4f}")
    print(f"  MSE: {val_mse:.4f}")
    print(f"  MAE: {val_mae:.4f}")
    print(f"  RMSE: {np.sqrt(val_mse):.4f}")
    
    # Summarize the diagnostics
    print("\n" + "-"*50)
    print("Diagnostic summary:")
    
    r2_diff = abs(train_r2 - val_r2)
    
    if train_r2 < 0.6:
        if val_r2 > 0.8:
            print("⚠️ Low training performance and good validation performance")
            print("  The training set may contain more noise or outliers")
        else:
            print("⚠️ Possible underfitting")
            print(f"  - Low training R²: {train_r2:.4f}")
    elif r2_diff > 0.15:
        print("⚠️ Possible overfitting")
        print(f"  - Training–validation R² gap: {r2_diff:.4f}")
    else:
        print("✅ Good model performance")
        print(f"  - Training R²: {train_r2:.4f}")
        print(f"  - Validation R²: {val_r2:.4f}")
        print(f"  - Difference: {r2_diff:.4f}")
    
    # Check whether the model meets the targets
    if val_r2 >= 0.8 and train_r2 >= 0.6:
        print("\n🎉 Targets met!")
        print(f"  - Validation R² >= 0.8: {val_r2:.4f}")
        print(f"  - Training R² >= 0.6: {train_r2:.4f}")
    elif val_r2 >= 0.8:
        print(f"\n⚠️ Validation met its target (R²={val_r2:.4f}). Training needs improvement (R²={train_r2:.4f}).")
    else:
        print(f"\n❌ Validation target missed (R²={val_r2:.4f} < 0.8)")
    
    return {
        'train_r2': train_r2, 'val_r2': val_r2,
        'train_mse': train_mse, 'val_mse': val_mse,
        'y_train_pred': y_train_pred_original,
        'y_val_pred': y_val_pred_original
    }

def plot_results(history, results, y_train_original, y_val_original):
    """Plot the results."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Read the training history
    if isinstance(history, list):
        plot_history = history[0]
    elif hasattr(history, 'models') and history.models:
        plot_history = history.models[0].history
    else:
        plot_history = history.history if hasattr(history, 'history') else history
    
    # 1. Loss curves
    ax = axes[0, 0]
    ax.plot(plot_history['d_loss'], label='Discriminator Loss', alpha=0.7)
    ax.plot(plot_history['g_loss'], label='Generator Loss', alpha=0.7)
    ax.plot(plot_history['reg_loss'], label='Regression Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training losses')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. R² curves
    ax = axes[0, 1]
    ax.plot(plot_history['train_r2'], label='Train R²', marker='o', markersize=2)
    ax.plot(plot_history['val_r2'], label='Validation R²', marker='s', markersize=2)
    ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target R²=0.8')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Min Train R²=0.6')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('R² Score')
    ax.set_title('R² over epochs')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. Training predictions vs. observations
    ax = axes[0, 2]
    ax.scatter(y_train_original, results['y_train_pred'], alpha=0.5, s=10)
    ax.plot([y_train_original.min(), y_train_original.max()], 
            [y_train_original.min(), y_train_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('Observed runout distance (m)')
    ax.set_ylabel('Predicted runout distance (m)')
    ax.set_title(f'Training set: R² = {results["train_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 4. Validation predictions vs. observations
    ax = axes[1, 0]
    ax.scatter(y_val_original, results['y_val_pred'], alpha=0.5, s=10, color='orange')
    ax.plot([y_val_original.min(), y_val_original.max()], 
            [y_val_original.min(), y_val_original.max()], 
            'r--', lw=2)
    ax.set_xlabel('Observed runout distance (m)')
    ax.set_ylabel('Predicted runout distance (m)')
    ax.set_title(f'Validation set: R² = {results["val_r2"]:.4f}')
    ax.grid(True, alpha=0.3)
    
    # 5. Training residual distribution
    ax = axes[1, 1]
    residuals_train = y_train_original.flatten() - results['y_train_pred'].flatten()
    ax.hist(residuals_train, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('Residual (m)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Training residuals (mean={np.mean(residuals_train):.2f})')
    ax.grid(True, alpha=0.3)
    
    # 6. Validation residual distribution
    ax = axes[1, 2]
    residuals_val = y_val_original.flatten() - results['y_val_pred'].flatten()
    ax.hist(residuals_val, bins=30, edgecolor='black', alpha=0.7, color='orange')
    ax.axvline(x=0, color='r', linestyle='--')
    ax.set_xlabel('Residual (m)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Validation residuals (mean={np.mean(residuals_val):.2f})')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('gan_outlier_removal_results.png', dpi=100, bbox_inches='tight')
    plt.show()

# ==================== 6. Main function ====================
def main(outlier_method='remove', iqr_multiplier=3, use_ensemble=True):
    """
    Run the runout distance prediction workflow.
    
    Parameters:
    - outlier_method: Choose 'remove' to drop outliers or 'clip' to cap them.
    - iqr_multiplier: Multiplier for the outlier bounds.
    - use_ensemble: Whether to train an ensemble.
    """
    # Select the computation device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    print(f"Outlier treatment: {outlier_method}")
    print(f"IQR multiplier: {iqr_multiplier}")
    print(f"Use ensemble: {use_ensemble}")
    print("="*50)
    
    # 1. Load the data and treat outliers
    X, y, column_names = load_and_preprocess_data(
        "imputed_soil_with_runout.csv",
        outlier_method=outlier_method,
        iqr_multiplier=iqr_multiplier
    )
    
    # 2. Scale the data
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)
    
    # 3. Split the training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.2, random_state=42
    )
    
    print(f"\nData split:")
    print(f"Training set shape: {X_train.shape}")
    print(f"Validation set shape: {X_val.shape}")
    print(f"Feature count: {X_train.shape[1]}")
    
    # 4. Select the model type
    if use_ensemble:
        print("\nTraining the optimized GAN ensemble...")
        model = OptimizedEnsembleGAN(
            n_models=3, 
            input_dim=X_train.shape[1],
            device=device
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    else:
        print("\nTraining a single optimized GAN...")
        model = OptimizedGANRegressor(
            input_dim=X_train.shape[1],
            device=device,
            weight_decay=5e-5
        )
        
        model.fit(
            X_train, y_train,
            X_val, y_val,
            epochs=500,
            batch_size=32,
            patience=50,
            verbose=True
        )
    
    # 5. Diagnose model performance
    results = diagnose_model(
        model, 
        X_train, y_train, 
        X_val, y_val,
        scaler_y
    )
    
    # 6. Plot the results
    y_train_original = scaler_y.inverse_transform(y_train)
    y_val_original = scaler_y.inverse_transform(y_val)
    plot_results(model, results, y_train_original, y_val_original)
    
    # 7. Report the final results
    print("\n" + "="*50)
    print("Final results")
    print("="*50)
    print(f"Outlier treatment: {outlier_method}")
    print(f"Rows after preprocessing: {X.shape[0]}")
    print(f"\nR² scores on the original scale:")
    print(f"  Training: {results['train_r2']:.4f}")
    print(f"  Validation: {results['val_r2']:.4f}")
    print(f"  Difference: {abs(results['train_r2'] - results['val_r2']):.4f}")
    # 8. Export predictions to CSV
    train_df = pd.DataFrame({
        "Dataset": "Train",
        "Actual": y_train_original.flatten(),
        "Predicted": results['y_train_pred'].flatten()
    })

    val_df = pd.DataFrame({
        "Dataset": "Validation",
        "Actual": y_val_original.flatten(),
        "Predicted": results['y_val_pred'].flatten()
    })

    output_df = pd.concat([train_df, val_df], ignore_index=True)
    output_df.to_csv("train_val_predictions.csv", index=False, encoding="utf-8-sig")
    print("\n✅ Predictions saved to train_val_predictions.csv")
    
    if results['val_r2'] >= 0.8 and results['train_r2'] >= 0.6:
        print("\n🎉 Both performance targets met!")
        print(f"  Validation R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  Training R² = {results['train_r2']:.4f} >= 0.6 ✓")
        print("  The model meets both configured R² targets")
    elif results['val_r2'] >= 0.8:
        print(f"\n⚠️ Only the validation target was met")
        print(f"  Validation R² = {results['val_r2']:.4f} >= 0.8 ✓")
        print(f"  Training R² = {results['train_r2']:.4f} < 0.6 ✗")
        print("\nSuggestions:")
        print("  1. Check the quality of the training data")
        print("  2. Consider data cleaning or feature engineering")
        print("  3. Adjust the model architecture or training strategy")
    else:
        print(f"\n❌ Validation target missed")
        print(f"  Validation R² = {results['val_r2']:.4f} < 0.8")
        print("\nSuggestions:")
        print("  1. Increase model capacity")
        print("  2. Collect more training data")
        print("  3. Try a different model architecture")
    
    return model, results

# Run the workflow
if __name__ == "__main__":
    # Adjustable parameters:
    # outlier_method: 'remove' to drop outliers or 'clip' to cap them
    # iqr_multiplier: 1.5 (strict), 3 (standard), or 5 (lenient)
    # use_ensemble: True for an ensemble or False for a single model
    
    model, results = main(
        outlier_method='clip',  # Clip outliers
        iqr_multiplier=1.5,         # Use an IQR multiplier of 1.5
        use_ensemble=True         # Use the ensemble
    )
    
    print("\nTraining complete. Figure saved as 'gan_outlier_removal_results.png'.")